# Task 6 &nbsp;|&nbsp; MC Dropout uncertainty on ViT-tiny

**Due 04 September.** Adds Monte-Carlo dropout at inference to the trained fold models, reports
**ECE and Brier**, and exposes the result through a versioned contract.

## What this notebook needs attached

| Dataset | Contains | Why |
|---|---|---|
| the **nodule cache** | `lidc_nodules_cache_v2.npz` | same 322 nodules, same folds |
| the **Task 05-01 output** | `vit_tiny_fold&lt;k&gt;_best.pt` &times; 5 (+ the result files) | the weights to sample |

Nothing else. This notebook **trains nothing**, so it needs no long GPU session &mdash; roughly
10 minutes of inference. A GPU makes it faster and is optional; running with no
accelerator costs none of your weekly GPU quota.

## The deliverable, and what is extra

Sections 1&ndash;7 are Task 6 as assigned: MC Dropout, the uncertainty decomposition, **ECE and
Brier**, and the validation-calibrated thresholds. They run with no API key and no network.

Sections 8&ndash;10 add the routing layer that the orchestrator will use. Section 9 calls Qwen3
through OpenRouter and is **optional** &mdash; if no key is supplied it is skipped with a message
and everything else still completes. The deliverable never depends on the LLM.

## Why the fold loop

Task 4/5 produced **five** models, one per CV fold. Every nodule must be sampled by the one model
that never saw its patient, because a model is confident on data it memorised. The checkpoint's
stored fingerprint is verified against this session before any GPU time is spent: if the fold
partition differs, the "held-out" nodules include ones those models trained on, and the resulting
uncertainty would look good and mean nothing.

## 1 &nbsp;Configuration

Everything that defines the protocol lives here, and the values that must match across notebooks are
hashed into a dataset fingerprint (§4). Nothing below this cell hardcodes a number that belongs here.

`BUILD_VERSION` is new: it is written into the cache so a notebook can tell whether a cache it has
been handed was built with the corrected through-plane geometry or the old index-based one. Without
it, an old cache would load silently and reintroduce the bug.

In [ ]:
import os, sys, re, json, math, time, random, warnings, glob, zipfile, shutil, hashlib
from pathlib import Path
warnings.filterwarnings("ignore")

BUILD_VERSION = "2.0-mm-throughplane"   # 1.x = old index-based neighbour slices (do not use)

CFG = dict(
    SEED = 42,
    MODEL = "vit_tiny",
    PRETRAINED = True,

    # ---------- data source ----------
    DATA_SOURCE = "auto",               # auto | cache | lidc_raw | phantom
    CACHE_NAME  = "lidc_nodules_cache_v2.npz",
    # A cache built by an older version has the old through-plane geometry. "auto" skips such a
    # cache and rebuilds. Set True only to deliberately re-run on the old data for comparison -
    # results are then NOT the corrected pipeline.
    ALLOW_OLD_CACHE = False,
    WORK        = "/kaggle/working",
    RAW_DIR     = "/kaggle/temp/lidc_raw",
    XML_DIR     = "/kaggle/temp/lidc_xml",

    # ---------- nodule extraction (PROTOCOL - hashed into the fingerprint) ----------
    N_PATIENTS   = 300,
    MIN_READERS  = 3,        # nodule must be marked by >= this many of the 4 radiologists
    DROP_SCORE_3 = True,     # median malignancy exactly 3 -> indeterminate -> excluded
    MATCH_MM     = 6.0,      # cross-reader nodule matching radius
    PATCH_MM     = 48.0,     # in-plane crop: 48mm x 48mm of anatomy, then resampled
    IMG_SIZE     = 224,
    N_SLICES     = 3,        # 2.5D context
    SLICE_MM     = 2.5,      # THE FIX: neighbour slices at +/- this many mm, not +/- 1 index
    HU_WINDOW    = (-1000.0, 400.0),

    # ---------- patient-level cross-validation ----------
    N_FOLDS = 5,
    VAL_FRAC_OF_TRAIN = 0.15,

    # ---------- training (fixed a priori - see audit A5) ----------
    EPOCHS = 30, BATCH_SIZE = 32, LR = 3e-4, WEIGHT_DECAY = 1e-4,
    DROPOUT_P = 0.30, DROPOUT_MODE = "head_and_stages",
    PATIENCE = 7, AMP = True, NUM_WORKERS = 2, CLASS_WEIGHTS = True,
    ALLOW_CPU_FALLBACK = True,

    VIT_BACKBONE = "vit_tiny_patch16_224",

    # ---------- TASK 6 ----------
    MC_PASSES = 50,                  # stochastic forward passes per nodule
    UQ_TARGET_SENSITIVITY = 0.95,    # thresholds calibrated on validation to reach this
    USE_LLM = True,                  # section 9; set False to skip the orchestrator entirely
    LLM_MAX_CALLS = 400,             # cap while experimenting; 322 covers the whole dataset

    # ---------- evaluation ----------
    TARGET_SPECIFICITIES = [0.90, 0.95, 0.99],
    TARGET_SENSITIVITY = 0.95,   # for the validation-selected clinical operating point (§7)
    N_BOOTSTRAP = 1000,
    N_ECE_BINS = 15,
)

CFG["OUT_DIR"]    = f"{CFG['WORK']}/task06_outputs"
CFG["CACHE_PATH"] = f"{CFG['WORK']}/{CFG['CACHE_NAME']}"
Path(CFG["OUT_DIR"]).mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["benign", "malignant"]
IS_SMOKE = False

# Only these affect which nodules exist and how they are split. Two runs that agree on all of them
# (and on the cache) are comparable with paired statistics; two that do not, are not.
PROTOCOL_KEYS = ["SEED", "N_PATIENTS", "MIN_READERS", "DROP_SCORE_3", "MATCH_MM", "PATCH_MM",
                 "IMG_SIZE", "N_SLICES", "SLICE_MM", "HU_WINDOW", "N_FOLDS", "VAL_FRAC_OF_TRAIN"]

print(f"Task 6  |  build {BUILD_VERSION}  |  model {CFG['MODEL']}  |  "
      f"T={CFG['MC_PASSES']} MC passes")
print(json.dumps({k: CFG[k] for k in PROTOCOL_KEYS}, indent=2, default=str))

### 1.1 &nbsp;Environment and GPU preflight

A PyTorch wheel only contains kernels for the GPU architectures it was compiled for. Kaggle's P100
is `sm_60`, which recent CUDA 12.x builds no longer ship - every CUDA call then fails, but only when
first made, which is deep inside training. The convolution below proves the device works before
anything expensive starts. With `ALLOW_CPU_FALLBACK` the run continues on CPU instead of stopping;
at this dataset size CPU is about 51 minutes for five folds, which is slow but perfectly usable.

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib.pyplot as plt

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| numpy", np.__version__)

def gpu_preflight():
    """Prove the GPU can run a convolution before the expensive work begins."""
    global DEVICE
    if not torch.cuda.is_available():
        print("No accelerator attached - running on CPU (~51 min for 5 folds at this size).")
        print("For a GPU: Session options -> Accelerator -> GPU T4 x2  (not P100).")
        return False
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mine, built = f"sm_{cap[0]}{cap[1]}", [a for a in torch.cuda.get_arch_list()
                                           if a.startswith("sm_")]
    print(f"GPU {name} ({mine}) | torch built for {', '.join(built)}")
    try:
        torch.nn.functional.conv2d(torch.zeros(1, 3, 8, 8, device="cuda"),
                                   torch.zeros(4, 3, 3, 3, device="cuda"))
        torch.cuda.synchronize()
        print("preflight OK - a real convolution ran on the device")
        return True
    except Exception as e:
        msg = (f"\n{'='*70}\nTHIS GPU CANNOT RUN THIS PYTORCH BUILD\n{'='*70}\n"
               f"  {name} is {mine}; this wheel has {', '.join(built)}\n"
               f"  {type(e).__name__}: {e}\n"
               f"  FIX: Session options -> Accelerator -> 'GPU T4 x2' (sm_75)\n{'='*70}")
        print(msg)
        if CFG["ALLOW_CPU_FALLBACK"]:
            DEVICE = torch.device("cpu")
            print("  ALLOW_CPU_FALLBACK is on - continuing on CPU.")
            return False
        raise RuntimeError(msg) from e

GPU_OK = gpu_preflight()
print("device:", DEVICE)

## 2 &nbsp;How the malignancy label is constructed

LIDC-IDRI has no label column. Up to four radiologists independently outlined each nodule and scored
it 1-5 for malignancy (1 = highly unlikely, 5 = highly suspicious). Producing a binary label takes
three decisions, and results are only comparable between studies that made them the same way.

| Decision | Choice | Reasoning |
|---|---|---|
| Which nodules to keep | marked by **&ge; 3** of 4 readers | single-reader marks are noisy and inflate scores |
| How to fuse the ratings | **median** | robust to one dissenting reader; the dominant convention |
| Median exactly 3 | **excluded** | indeterminate is neither class; keeping it corrupts both |

Label: median &lt; 3 &rarr; benign, median &gt; 3 &rarr; malignant.

**A consequence worth stating plainly.** With four readers the median can be 2.5 or 3.5. Those
nodules are one reader away from the exclusion rule, and in the previous run they were 115 of 322
(36%) of the dataset. They are kept - dropping them is not the convention - but §9 reports what
happens if they are removed, because a reviewer will ask whether the result rests on them.

The other eight characteristics (spiculation, margin, subtlety, lobulation, calcification, texture,
sphericity, internal structure) are cached too: they are needed for the false-negative analysis in
§8 and for Task 8's concept-based explainability.

### Matching one nodule across four readers

The readers worked blind to each other, so reader 1's "nodule 3" and reader 4's "nodule 1" may be
the same lesion. They are grouped by centroid proximity in millimetres before the ratings are
fused, with one constraint a plain nearest-neighbour pass gets wrong: **a cluster may contain at
most one nodule per reader.** Without it, two distinct lesions that one reader marked close together
merge into a single cluster, its reader count becomes meaningless, and the &ge;3-reader filter
silently stops working.

In [ ]:
import xml.etree.ElementTree as ET
from collections import defaultdict

def _tag(e):
    """LIDC XML namespaces vary between files - compare on the local name only."""
    return e.tag.split("}")[-1]

CHARACTERISTIC_KEYS = ["subtlety", "internalStructure", "calcification", "sphericity",
                       "margin", "lobulation", "spiculation", "texture", "malignancy"]

def parse_lidc_xml(xml_path):
    """One LIDC XML -> (series_uid, [per-reader nodule lists]).

    Nodules without a <characteristics> block are the <3mm marks: they carry no malignancy
    rating, so they cannot be labelled and are dropped here.
    """
    root = ET.parse(xml_path).getroot()
    series_uid = None
    for e in root.iter():
        if _tag(e) in ("SeriesInstanceUid", "SeriesInstanceUID"):
            series_uid = (e.text or "").strip(); break

    sessions = []
    for sess in root.iter():
        if _tag(sess) != "readingSession":
            continue
        nodules = []
        for nod in sess:
            if _tag(nod) != "unblindedReadNodule":
                continue
            chars, rois = {}, []
            for child in nod:
                t = _tag(child)
                if t == "characteristics":
                    for c in child:
                        try:
                            chars[_tag(c)] = int(float((c.text or "").strip()))
                        except (TypeError, ValueError):
                            pass
                elif t == "roi":
                    z, pts, included = None, [], True
                    for r in child:
                        rt = _tag(r)
                        if rt == "imageZposition":
                            z = float(r.text)
                        elif rt == "inclusion":
                            included = (r.text or "").strip().upper() == "TRUE"
                        elif rt == "edgeMap":
                            x = y = None
                            for e2 in r:
                                if _tag(e2) == "xCoord":   x = int(float(e2.text))
                                elif _tag(e2) == "yCoord": y = int(float(e2.text))
                            if x is not None and y is not None:
                                pts.append((x, y))
                    if pts and included:
                        rois.append(dict(z=z, pts=np.asarray(pts, dtype=np.float32)))
            if rois and "malignancy" in chars:
                nodules.append(dict(characteristics=chars, rois=rois))
        sessions.append(nodules)
    return series_uid, sessions


def nodule_geometry(nodule):
    """Centroid (col, row, z_mm) plus in-plane extents in pixels.

    Returns the bounding-box width and height separately rather than a single "diameter":
    the previous run reported the box DIAGONAL as diam_mm, which overstates a round nodule's
    size by a factor of ~1.41 and made patch-overflow look worse than it is.
    """
    pts = np.concatenate([r["pts"] for r in nodule["rois"]], axis=0)
    zs = [r["z"] for r in nodule["rois"] if r["z"] is not None]
    w = h = 0.0
    for r in nodule["rois"]:
        p = r["pts"]
        if len(p) > 1:
            w = max(w, float(np.ptp(p[:, 0]))); h = max(h, float(np.ptp(p[:, 1])))
    return (float(pts[:, 0].mean()), float(pts[:, 1].mean()),
            float(np.mean(zs)) if zs else None, w, h)


def cluster_nodules_across_readers(sessions, spacing_xy, match_mm):
    """Group per-reader marks into physical lesions, at most one mark per reader per cluster."""
    items = []
    for r_idx, nodules in enumerate(sessions):
        for nod in nodules:
            cx, cy, cz, w, h = nodule_geometry(nod)
            if cz is None:
                continue
            items.append(dict(reader=r_idx, nodule=nod, cx=cx, cy=cy, cz=cz, w=w, h=h))

    items.sort(key=lambda d: (d["cz"], d["cx"], d["cy"], d["reader"]))   # deterministic
    sx, sy = spacing_xy
    clusters = []
    for it in items:
        best, best_d = None, np.inf
        for cl in clusters:
            if any(m["reader"] == it["reader"] for m in cl):
                continue                                   # reader already represented
            d = math.sqrt(((it["cx"] - np.mean([m["cx"] for m in cl])) * sx) ** 2 +
                          ((it["cy"] - np.mean([m["cy"] for m in cl])) * sy) ** 2 +
                          (it["cz"] - np.mean([m["cz"] for m in cl])) ** 2)
            if d < best_d:
                best, best_d = cl, d
        if best is not None and best_d <= match_mm:
            best.append(it)
        else:
            clusters.append([it])
    return clusters


def fuse_cluster_labels(cluster, min_readers, drop_score_3):
    """Cluster -> one labelled nodule, or None if it fails the protocol."""
    readers = {m["reader"] for m in cluster}
    if len(readers) < min_readers:
        return None
    scores = defaultdict(list)
    for m in cluster:
        for k in CHARACTERISTIC_KEYS:
            if k in m["nodule"]["characteristics"]:
                scores[k].append(m["nodule"]["characteristics"][k])
    if not scores.get("malignancy"):
        return None

    mal = float(np.median(scores["malignancy"]))
    if drop_score_3 and abs(mal - 3.0) < 1e-9:
        return None

    return dict(
        n_readers=len(readers), malignancy=mal, label=int(mal > 3.0),
        # spread of the 3-4 human ratings: measured ambiguity, used in the FN analysis and Task 6
        mal_std=float(np.std(scores["malignancy"])),
        cx=float(np.mean([m["cx"] for m in cluster])),
        cy=float(np.mean([m["cy"] for m in cluster])),
        cz=float(np.mean([m["cz"] for m in cluster])),
        bbox_w_px=float(np.mean([m["w"] for m in cluster])),
        bbox_h_px=float(np.mean([m["h"] for m in cluster])),
        **{f"c_{k}": float(np.median(v)) for k, v in scores.items()},
    )

## 3 &nbsp;Patch extraction &mdash; and the through-plane fix

This is where the review question "*is the 3-slice input centred consistently on the nodule?*" is
answered, and where the previous implementation was wrong.

### In-plane: correct before, unchanged

The crop is **48 mm &times; 48 mm of anatomy**, converted to pixels per scan using that scan's
`PixelSpacing`, then resampled to 224 &times; 224. Pixel spacing across LIDC varies roughly
0.5-0.9 mm, so a fixed *pixel* crop would contain ~40% more tissue on one scan than another and the
network would partly learn scanner configuration. Sizing in millimetres removes that.

Padding is applied **before** cropping, so a nodule against the chest wall still yields a full-size
patch with the nodule at the centre. Cropping first and padding after would shift it off-centre.

### Through-plane: this was the bug

The two neighbouring slices were taken at **&plusmn;1 slice index**. Slice thickness in LIDC ranges
roughly 0.6-5.0 mm, so:

| Scan thickness | Old context window | What the 3 channels actually held |
|---|---|---|
| 0.6 mm | &plusmn;0.6 mm | three near-identical slices |
| 2.5 mm | &plusmn;2.5 mm | the intended 2.5D context |
| 5.0 mm | &plusmn;5.0 mm | three quite different levels |

So the amount of through-plane context was a function of the acquisition protocol, which varies by
institution and scanner - a confound sitting in the one dimension nobody was auditing.

**The fix:** neighbours are selected at a fixed physical offset (`SLICE_MM` = 2.5 mm) by finding the
nearest available slice to `z_centroid &plusmn; SLICE_MM`. On a thick-slice scan the nearest slice may
still be several mm away, and that residual cannot be removed without interpolating - so the
**achieved** offset is recorded per nodule and audited in A3, rather than assumed.

### Provenance recorded per nodule

The previous cache stored no acquisition geometry, so these questions could not be answered from it
at all. Each nodule now carries `spacing_x`, `spacing_y`, `slice_thickness`, the achieved
through-plane offsets, and whether the lesion overflowed the patch.

In [ ]:
def load_ct_series(series_dir):
    """Read one DICOM CT series -> HU volume sorted along z, plus acquisition geometry."""
    import pydicom
    slices = []
    for p in glob.glob(os.path.join(series_dir, "**", "*.dcm"), recursive=True):
        try:
            ds = pydicom.dcmread(p, force=True)
            if hasattr(ds, "PixelData") and hasattr(ds, "ImagePositionPatient") \
                    and getattr(ds, "Modality", "CT") == "CT":
                slices.append(ds)
        except Exception:
            continue
    if len(slices) < 10:
        return None

    slices.sort(key=lambda d: float(d.ImagePositionPatient[2]))
    ref = slices[0]
    spacing = [float(v) for v in getattr(ref, "PixelSpacing", [1.0, 1.0])]
    vol = np.stack([s.pixel_array.astype(np.float32) for s in slices], axis=0)
    vol = vol * float(getattr(ref, "RescaleSlope", 1.0)) + float(getattr(ref, "RescaleIntercept", 0.0))
    vol[vol < -2000] = -1000.0          # out-of-field padding -> air

    z = np.array([float(s.ImagePositionPatient[2]) for s in slices])
    return dict(vol=vol, z_pos=z,
                spacing=(spacing[1], spacing[0]),                    # (x, y) mm/pixel
                thickness=float(getattr(ref, "SliceThickness", np.nan)),
                z_step=float(np.median(np.diff(z))) if len(z) > 1 else np.nan,
                patient_id=str(getattr(ref, "PatientID", "")),
                series_uid=str(getattr(ref, "SeriesInstanceUID", "")))


def extract_patch(series, nod, cfg):
    """Nodule -> (n_slices, IMG_SIZE, IMG_SIZE) uint8 patch + the geometry actually achieved.

    In-plane size and through-plane offsets are both specified in millimetres. The achieved
    offsets are returned because on a thick-slice scan the nearest slice to +/-SLICE_MM can be
    some way off, and that residual should be visible rather than silently absorbed.
    """
    import cv2
    vol, z_pos = series["vol"], series["z_pos"]
    sx, sy = series["spacing"]
    lo, hi = cfg["HU_WINDOW"]

    half_x = max(2, int(round((cfg["PATCH_MM"] / 2.0) / sx)))
    half_y = max(2, int(round((cfg["PATCH_MM"] / 2.0) / sy)))
    cx, cy = int(round(nod["cx"])), int(round(nod["cy"]))

    n = cfg["N_SLICES"]
    wanted = [0.0] if n == 1 else [(i - n // 2) * cfg["SLICE_MM"] for i in range(n)]

    out, achieved, chosen = [], [], []
    for off_mm in wanted:
        target = nod["cz"] + off_mm
        # Nearest slice to the target. On a regular grid the target often falls exactly halfway
        # between two slices; np.argmin would then always take the lower index, which makes the
        # context window lopsided (e.g. -3mm above but +2mm below the centroid). Break ties
        # toward the centroid so the sampling stays symmetric.
        d = np.abs(z_pos - target)
        cand = np.flatnonzero(d <= d.min() + 1e-9)
        zi = int(cand[np.argmin(np.abs(z_pos[cand] - nod["cz"]))])
        chosen.append(zi)
        achieved.append(float(z_pos[zi] - nod["cz"]))

        # pad first so a peripheral nodule still lands in the centre of a full-size crop
        pad_y, pad_x = half_y + 1, half_x + 1
        padded = np.pad(vol[zi], ((pad_y, pad_y), (pad_x, pad_x)),
                        mode="constant", constant_values=lo)
        crop = padded[cy + pad_y - half_y: cy + pad_y + half_y,
                      cx + pad_x - half_x: cx + pad_x + half_x]
        if crop.size == 0:
            return None, None
        win = ((np.clip(crop, lo, hi) - lo) / (hi - lo) * 255.0).astype(np.uint8)
        out.append(cv2.resize(win, (cfg["IMG_SIZE"],) * 2, interpolation=cv2.INTER_LINEAR))

    geom = dict(spacing_x=sx, spacing_y=sy,
                slice_thickness=series["thickness"], z_step=series["z_step"],
                offset_wanted_mm=json.dumps([round(v, 3) for v in wanted]),
                offset_achieved_mm=json.dumps([round(v, 3) for v in achieved]),
                offset_error_mm=float(np.max(np.abs(np.array(achieved) - np.array(wanted)))),
                # On a thick-slice scan no slice may exist near +/-SLICE_MM, so the three
                # channels can collapse to the same image. That is honest - the context simply
                # is not in the data - but it must be visible, not silent.
                n_distinct_slices=int(len(set(chosen))),
                nodule_w_mm=nod["bbox_w_px"] * sx, nodule_h_mm=nod["bbox_h_px"] * sy,
                overflows_patch=bool(max(nod["bbox_w_px"] * sx,
                                         nod["bbox_h_px"] * sy) > cfg["PATCH_MM"]))
    return np.stack(out, axis=0), geom

### 3.1 &nbsp;Downloading LIDC-IDRI

`tcia_utils` fetches the CT series. The nodule annotations are a **separate XML package** - the
original Task 3 run failed silently here, reporting `no annotations found`, because it downloaded
images only. If the XML cannot be fetched the cell raises with instructions rather than continuing
label-free.

In [ ]:
def download_lidc_dicom(n_patients, out_dir):
    """One CT series per patient for the first n_patients, deterministically selected."""
    try:
        from tcia_utils import nbia
    except ImportError:
        get_ipython().system("pip install -q tcia_utils")
        from tcia_utils import nbia

    os.makedirs(out_dir, exist_ok=True)
    df = pd.DataFrame(nbia.getSeries(collection="LIDC-IDRI"))
    df = df[df["Modality"] == "CT"].copy()                  # LIDC also contains DX/CR
    df["ImageCount"] = pd.to_numeric(df["ImageCount"], errors="coerce").fillna(0)
    df = (df.sort_values(["PatientID", "ImageCount"], ascending=[True, False])
            .groupby("PatientID", as_index=False).first().sort_values("PatientID"))
    sel = df.head(n_patients)
    print(f"LIDC-IDRI: {df.PatientID.nunique()} patients with CT | selecting {len(sel)}")
    print(f"~{0.9 * sel.ImageCount.sum() / 1000:.1f} GB, {int(sel.ImageCount.sum())} slices")
    nbia.downloadSeries(sel.to_dict("records"), path=out_dir)
    return sel


LIDC_XML_URLS = [
    "https://www.cancerimagingarchive.net/wp-content/uploads/LIDC-XML-only.zip",
    "https://wiki.cancerimagingarchive.net/download/attachments/1966254/LIDC-XML-only.zip",
]

def download_lidc_xml(xml_dir):
    """Fetch + unzip the annotation package. Raises loudly rather than proceeding unlabelled."""
    import requests
    os.makedirs(xml_dir, exist_ok=True)
    if glob.glob(os.path.join(xml_dir, "**", "*.xml"), recursive=True):
        print("XML annotations already present."); return xml_dir

    zp = os.path.join(xml_dir, "LIDC-XML-only.zip")
    for url in LIDC_XML_URLS:
        try:
            print("Trying", url)
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                with open(zp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(zp) as z:
                z.extractall(xml_dir)
            os.remove(zp)
            print(f"Extracted {len(glob.glob(os.path.join(xml_dir,'**','*.xml'), recursive=True))} XMLs")
            return xml_dir
        except Exception as e:
            print("  failed:", type(e).__name__, e)

    raise RuntimeError(
        "Could not fetch the LIDC XML annotation package.\n"
        "MANUAL FIX (once): download 'LIDC-XML-only.zip' from the LIDC-IDRI page on "
        "cancerimagingarchive.net, upload it as a Kaggle Dataset, attach it, and set "
        "CFG['XML_DIR'] to that /kaggle/input path.\n"
        "Without these XMLs there are no malignancy labels and no supervised task.")


def index_xml_by_series(xml_dir):
    idx, files = {}, glob.glob(os.path.join(xml_dir, "**", "*.xml"), recursive=True)
    for p in files:
        try:
            uid, _ = parse_lidc_xml(p)
            if uid:
                idx[uid] = p
        except Exception:
            continue
    print(f"Indexed {len(idx)} / {len(files)} XMLs by SeriesInstanceUID")
    return idx

### 3.2 &nbsp;Building the cache

Run once. Save the resulting `.npz` as a Kaggle Dataset and every later run - Task 5, Task 6, any
re-run - attaches it and skips the download.

Rows are written in a **canonical order** (patient, then series, then nodule position) rather than
filesystem traversal order. The fold assignment is derived from row order, so without this two
machines could download identical data and still produce different folds.

In [ ]:
def build_lidc_cache(cfg):
    """Raw LIDC -> labelled patch cache with full geometric provenance."""
    t0 = time.time()
    download_lidc_dicom(cfg["N_PATIENTS"], cfg["RAW_DIR"])
    xml_index = index_xml_by_series(download_lidc_xml(cfg["XML_DIR"]))

    patches, rows, stats = [], [], defaultdict(int)
    series_dirs = sorted(d for d in glob.glob(os.path.join(cfg["RAW_DIR"], "**"), recursive=True)
                         if os.path.isdir(d) and glob.glob(os.path.join(d, "*.dcm")))
    print(f"{len(series_dirs)} series directories with DICOM")

    for i, sdir in enumerate(series_dirs, 1):
        series = load_ct_series(sdir)
        if series is None:
            stats["series_unreadable"] += 1; continue
        suid = series["series_uid"] or os.path.basename(sdir)
        if suid not in xml_index:
            stats["series_no_annotation"] += 1; continue

        _, sessions = parse_lidc_xml(xml_index[suid])
        clusters = cluster_nodules_across_readers(sessions, series["spacing"], cfg["MATCH_MM"])
        stats["clusters_found"] += len(clusters)

        kept = 0
        for cl in clusters:
            nod = fuse_cluster_labels(cl, cfg["MIN_READERS"], cfg["DROP_SCORE_3"])
            if nod is None:
                stats["rejected_by_protocol"] += 1; continue
            patch, geom = extract_patch(series, nod, cfg)
            if patch is None:
                stats["patch_failed"] += 1; continue
            nod.update(geom, patient_id=series["patient_id"] or suid, series_uid=suid)
            patches.append(patch); rows.append(nod); kept += 1

        stats["series_used"] += 1
        if kept:
            print(f"  [{i}/{len(series_dirs)}] {series['patient_id']}: +{kept} "
                  f"(total {len(patches)})")
        del series                      # a 512x512x140 float32 volume is ~150 MB

    if not patches:
        raise RuntimeError(f"No nodules extracted. Diagnostics: {dict(stats)}")

    # canonical order - row order must depend on the data, not on disk layout
    order = sorted(range(len(rows)), key=lambda i: (
        str(rows[i]["patient_id"]), str(rows[i]["series_uid"]),
        round(rows[i]["cz"], 2), round(rows[i]["cy"], 2), round(rows[i]["cx"], 2)))
    patches = [patches[i] for i in order]; rows = [rows[i] for i in order]

    X, meta = np.stack(patches).astype(np.uint8), pd.DataFrame(rows)
    np.savez_compressed(
        cfg["CACHE_PATH"], patches=X,
        labels=meta["label"].to_numpy(np.int64),
        malignancy=meta["malignancy"].to_numpy(np.float32),
        patient_id=meta["patient_id"].to_numpy(object),
        meta_json=json.dumps(meta.to_dict("list"), default=str),
        cfg_json=json.dumps({**{k: cfg[k] for k in PROTOCOL_KEYS},
                             "BUILD_VERSION": BUILD_VERSION,
                             "diagnostics": dict(stats)}, default=str))

    print(f"\nBuilt in {(time.time()-t0)/60:.1f} min: {X.shape[0]} nodules from "
          f"{meta.patient_id.nunique()} patients -> {cfg['CACHE_PATH']} "
          f"({os.path.getsize(cfg['CACHE_PATH'])/1e6:.0f} MB)")
    print(f"diagnostics: {dict(stats)}")
    print(">>> Save a Version, then publish this .npz as a Kaggle Dataset.")
    return cfg["CACHE_PATH"]


def build_phantom_cache(cfg, n_patients=60, per_patient=6):
    """Synthetic stand-in so the pipeline stays runnable without LIDC. Results are NOT reportable."""
    import cv2
    rng = np.random.default_rng(cfg["SEED"]); S, K = cfg["IMG_SIZE"], cfg["N_SLICES"]
    X, rows = [], []
    for p in range(n_patients):
        for _ in range(per_patient):
            y = int(rng.random() < 0.45)
            base = cv2.GaussianBlur(rng.normal(.25, .06, (S, S)).astype(np.float32), (0, 0), 6)
            yy, xx = np.mgrid[0:S, 0:S].astype(np.float32)
            cxp, cyp = S / 2 + rng.normal(0, 4, 2); r = S * rng.uniform(.10, .20)
            th = np.arctan2(yy - cyp, xx - cxp)
            rr = r * (1 + (0.32 if y else 0.06) * np.sin(int(rng.integers(6, 12)) * th))
            blob = np.clip((rr - np.hypot(xx - cxp, yy - cyp)) / (.18 * r), 0, 1)
            img = np.clip(cv2.GaussianBlur(base + blob * rng.uniform(.45, .70), (0, 0), 1), 0, 1)
            X.append((np.stack([np.clip(img + rng.normal(0, .012, (S, S)), 0, 1)
                                for _ in range(K)]) * 255).astype(np.uint8))
            rows.append(dict(patient_id=f"PHANTOM-{p:04d}", series_uid=f"S{p}",
                             malignancy=float(rng.choice([4., 5.]) if y else rng.choice([1., 2.])),
                             label=y, n_readers=4, mal_std=float(rng.uniform(0, 1)),
                             cx=S/2, cy=S/2, cz=0.0, bbox_w_px=r, bbox_h_px=r,
                             spacing_x=0.7, spacing_y=0.7, slice_thickness=2.5, z_step=2.5,
                             offset_wanted_mm="[-2.5, 0.0, 2.5]",
                             offset_achieved_mm="[-2.5, 0.0, 2.5]", offset_error_mm=0.0,
                             n_distinct_slices=3,
                             nodule_w_mm=r*0.7, nodule_h_mm=r*0.7, overflows_patch=False,
                             **{f"c_{k}": 3.0 for k in CHARACTERISTIC_KEYS}))
    meta = pd.DataFrame(rows)
    np.savez_compressed(cfg["CACHE_PATH"], patches=np.stack(X),
                        labels=meta["label"].to_numpy(np.int64),
                        malignancy=meta["malignancy"].to_numpy(np.float32),
                        patient_id=meta["patient_id"].to_numpy(object),
                        meta_json=json.dumps(meta.to_dict("list"), default=str),
                        cfg_json=json.dumps({"PHANTOM": True,
                                             "BUILD_VERSION": BUILD_VERSION}))
    print(f"PHANTOM cache: {len(X)} patches / {n_patients} pseudo-patients")
    return cfg["CACHE_PATH"]


def cache_build_version(path):
    """Which builder produced this cache. Old caches predate the field entirely."""
    try:
        with np.load(path, allow_pickle=True) as z:
            if not {"patches", "labels", "patient_id"} <= set(z.files):
                return None
            cfg = json.loads(str(z["cfg_json"])) if "cfg_json" in z.files else {}
            return cfg.get("BUILD_VERSION", "1.x (pre-fix)")
    except Exception:
        return None


def find_input_cache(name, cfg):
    """Find an attached cache built by THIS version.

    The generic *.npz sweep is what makes attaching the wrong dataset easy, so version is checked
    before a cache is accepted rather than after it is loaded. A mismatched cache is named and
    skipped, so the reason for the rebuild is obvious in the log.
    """
    seen, usable = [], []
    for p in (glob.glob(f"/kaggle/input/**/{name}", recursive=True)
              + glob.glob("/kaggle/input/**/*.npz", recursive=True)):
        if p in [s for s, _ in seen]:
            continue
        v = cache_build_version(p)
        if v is None:
            continue                                   # not a nodule cache at all
        seen.append((p, v))
        if v == BUILD_VERSION or cfg.get("ALLOW_OLD_CACHE", False):
            usable.append((p, v))

    for p, v in seen:
        mark = "USABLE" if (p, v) in usable else "SKIPPED - wrong build version"
        print(f"  found cache {os.path.basename(p)}  build={v}  -> {mark}")

    if usable:
        p, v = usable[0]
        if v != BUILD_VERSION:
            print(f"  ALLOW_OLD_CACHE is on: using a '{v}' cache deliberately.")
        return p
    if seen:
        print(f"  No attached cache was built by '{BUILD_VERSION}'. Rebuilding from raw LIDC.")
        print(f"  (Set CFG['ALLOW_OLD_CACHE']=True to reuse the old one for comparison instead.)")
    return None


def resolve_data(cfg):
    global IS_SMOKE
    src = cfg["DATA_SOURCE"]
    if src in ("auto", "cache"):
        p = find_input_cache(cfg["CACHE_NAME"], cfg)
        if p is None and os.path.exists(cfg["CACHE_PATH"]):
            v = cache_build_version(cfg["CACHE_PATH"])
            if v == BUILD_VERSION or cfg.get("ALLOW_OLD_CACHE", False):
                p = cfg["CACHE_PATH"]
        if p:
            print("Using cache:", p); return p
        if src == "cache":
            raise FileNotFoundError(
                f"DATA_SOURCE='cache' but no cache built by '{BUILD_VERSION}' was found.\n"
                f"Either attach one, set DATA_SOURCE='lidc_raw' to build it, or set "
                f"CFG['ALLOW_OLD_CACHE']=True to reuse an older cache knowingly.")
    if src in ("auto", "lidc_raw"):
        try:
            return build_lidc_cache(cfg)
        except Exception as e:
            print(f"\n{'!'*70}\nRAW BUILD FAILED: {type(e).__name__}: {e}\n{'!'*70}")
            if src == "lidc_raw":
                raise
    print(f"\n{'='*70}\nFALLING BACK TO PHANTOM DATA - RESULTS NOT REPORTABLE\n{'='*70}")
    IS_SMOKE = True
    return build_phantom_cache(cfg)


CACHE = resolve_data(CFG)

In [ ]:
with np.load(CACHE, allow_pickle=True) as z:
    PATCHES    = z["patches"]
    LABELS     = z["labels"].astype(np.int64)
    MALIGNANCY = z["malignancy"].astype(np.float32)
    PATIENT_ID = np.asarray([str(v) for v in z["patient_id"]])
    BUILD_CFG  = json.loads(str(z["cfg_json"]))
    META       = pd.DataFrame(json.loads(str(z["meta_json"])))

IS_SMOKE = IS_SMOKE or bool(BUILD_CFG.get("PHANTOM", False))
CACHE_BUILD = BUILD_CFG.get("BUILD_VERSION", "1.x (pre-fix)")

print(f"patches {PATCHES.shape} {PATCHES.dtype} | {len(LABELS)} nodules, "
      f"{len(np.unique(PATIENT_ID))} patients")
print(f"benign {int((LABELS==0).sum())} | malignant {int((LABELS==1).sum())} "
      f"({100*LABELS.mean():.1f}% positive)")
print(f"cache build version: {CACHE_BUILD}")
print(f"SMOKE: {IS_SMOKE}")

# The whole point of this notebook is the corrected through-plane geometry. An old cache would
# load without complaint and silently reintroduce the defect, so refuse to be quiet about it.
HAS_GEOMETRY = "offset_error_mm" in META.columns
if CACHE_BUILD != BUILD_VERSION:
    print(f"\n{'!'*74}")
    print(f"WARNING: this cache was built by version '{CACHE_BUILD}', not '{BUILD_VERSION}'.")
    print(f"(You are seeing this because ALLOW_OLD_CACHE is on, or the cache was in the working")
    print(f"directory. Otherwise the resolver would have rebuilt instead.)")
    print("Its neighbour slices were taken at +/-1 slice INDEX, so the 2.5D context is not")
    print("physically consistent across scans - the defect this notebook exists to fix.")
    print("Results below are still internally valid, but do NOT present them as the corrected")
    print("pipeline. Rebuild with DATA_SOURCE='lidc_raw' for that.")
    print(f"{'!'*74}")

### Audit A3 &nbsp;|&nbsp; Patch geometry

Evidence for "*how was the 48 mm patch extracted*" and "*is the 3-slice input centred consistently*".

- **In-plane**: the pixel-spacing spread shows how much a fixed-pixel crop would have varied.
- **Through-plane**: `offset_error_mm` is how far the chosen neighbour slice actually sat from the
  requested &plusmn;2.5 mm. Zero means exact; large values mean a thick-slice scan where no closer
  slice exists. This residual is a property of the data, not of the code, and is reported rather
  than hidden.
- **Overflow**: nodules whose in-plane extent exceeds 48 mm are clipped. LIDC's inclusion criterion
  is 3-30 mm so this should be rare; the count is reported so "rare" is a number.

In [ ]:
if not HAS_GEOMETRY:
    print("This cache predates geometric provenance (v1.x) - audit A3 cannot be run on it.\n"
          "The v1.x builder recorded no spacing or thickness, which is itself part of why\n"
          "the review's question could not be answered from the previous artefacts.")
else:
    g = META
    print("IN-PLANE pixel spacing (mm/px)")
    print(f"  x: min {g.spacing_x.min():.3f}  max {g.spacing_x.max():.3f}  "
          f"median {g.spacing_x.median():.3f}")
    print(f"  a fixed-pixel crop would have spanned "
          f"{g.spacing_x.min()*CFG['IMG_SIZE']:.0f}-{g.spacing_x.max()*CFG['IMG_SIZE']:.0f} mm "
          f"of anatomy; the mm-based crop spans {CFG['PATCH_MM']:.0f} mm for every nodule.")

    print("\nTHROUGH-PLANE")
    print(f"  slice thickness: min {g.slice_thickness.min():.2f}  max "
          f"{g.slice_thickness.max():.2f}  median {g.slice_thickness.median():.2f} mm")
    print(f"  requested neighbour offset: +/-{CFG['SLICE_MM']} mm")
    print(f"  achieved offset error: median {g.offset_error_mm.median():.2f}  "
          f"p95 {g.offset_error_mm.quantile(.95):.2f}  max {g.offset_error_mm.max():.2f} mm")
    bad = int((g.offset_error_mm > CFG["SLICE_MM"]).sum())
    print(f"  nodules where no slice exists within {CFG['SLICE_MM']} mm of the target: "
          f"{bad} ({100*bad/len(g):.1f}%)  <- thick-slice scans, irreducible without interpolation")
    if "n_distinct_slices" in g.columns:
        deg = int((g.n_distinct_slices < CFG["N_SLICES"]).sum())
        print(f"  nodules whose {CFG['N_SLICES']} channels collapse to fewer distinct slices: "
              f"{deg} ({100*deg/len(g):.1f}%)")
        print(f"    on those, the 2.5D input carries no real through-plane context. The context")
        print(f"    is absent from the acquisition, not discarded by the code - but if this "
              f"fraction")
        print(f"    is large, report it as a dataset limitation rather than leaving it implicit.")
    print(f"  under the OLD +/-1-index rule the context window would have been "
          f"+/-{g.slice_thickness.min():.1f} to +/-{g.slice_thickness.max():.1f} mm "
          f"({g.slice_thickness.max()/max(g.slice_thickness.min(),1e-6):.1f}x variation)")

    print("\nNODULE SIZE vs PATCH")
    ext = np.maximum(g.nodule_w_mm, g.nodule_h_mm)
    print(f"  in-plane extent: median {ext.median():.1f}  p95 {ext.quantile(.95):.1f}  "
          f"max {ext.max():.1f} mm")
    print(f"  exceeding the {CFG['PATCH_MM']:.0f} mm patch: {int(g.overflows_patch.sum())} "
          f"({100*g.overflows_patch.mean():.1f}%) - these are clipped")

### Audit A4 &nbsp;|&nbsp; Label construction

Evidence for "*how were the malignancy labels constructed*". Each protocol rule is a filter, and
each filter's effect is counted here, so the path from raw reader marks to the final n is explicit.

In [ ]:
print(f"FINAL DATASET: n = {len(LABELS)} nodules from {len(np.unique(PATIENT_ID))} patients")
print(f"  benign {int((LABELS==0).sum())}  |  malignant {int((LABELS==1).sum())}")
print("  (This is the number evaluated. Every table below uses it; nothing is extrapolated.)")

if "diagnostics" in BUILD_CFG:
    d = BUILD_CFG["diagnostics"]
    print(f"\nBUILD FUNNEL")
    print(f"  reader-clusters found      {d.get('clusters_found','?')}")
    print(f"  rejected by protocol       {d.get('rejected_by_protocol','?')}  "
          f"(<{CFG['MIN_READERS']} readers, or median malignancy exactly 3)")
    print(f"  patch extraction failed    {d.get('patch_failed',0)}")
    print(f"  -> kept                    {len(LABELS)}")

print(f"\nREADER AGREEMENT")
print(f"  readers per nodule: {META.n_readers.value_counts().sort_index().to_dict()}")
print(f"  SD of the 3-4 malignancy ratings: median {META.mal_std.median():.2f} "
      f"(0 = unanimous)")

print(f"\nMEDIAN MALIGNANCY -> LABEL")
for v in sorted(META.malignancy.unique()):
    n = int((META.malignancy == v).sum())
    lab = CLASS_NAMES[int(v > 3.0)]
    flag = "   <- one reader from the exclusion rule" if abs(v - 3.0) == 0.5 else ""
    print(f"  {v:>3} -> {lab:<9} {n:4d}{flag}")

boundary = META.malignancy.isin([2.5, 3.5])
print(f"\n  nodules at the 2.5/3.5 boundary: {int(boundary.sum())} "
      f"({100*boundary.mean():.0f}% of the dataset)")
print("  These are kept, as convention requires. Section 9 reports the result without them.")

## 4 &nbsp;Patient-level folds, and proof that they are separated

Evidence for two review questions: "*were the patient-level folds genuinely separated*" and
"*did multiple nodules from the same patient stay in the same fold*".

**Why it matters here specifically.** In the previous dataset 75 of 187 patients contributed more
than one nodule, and one contributed nine. Nodules from one patient share a scanner, a
reconstruction kernel and the same anatomy, so a nodule-level split lets a model recognise the
patient rather than the pathology - worth several points of illusory AUROC.

`StratifiedGroupKFold` keeps whole patients together while balancing the class ratio across folds.
Each fold then carves a validation slice out of its *training* pool for early stopping; validation
is never the fold being scored, because selecting the epoch on the scored fold would leak it.

Audit A1 below does not assert separation - it enumerates every patient and every fold pair and
prints the actual overlap counts.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def patient_level_folds(labels, groups, n_folds, val_frac, seed):
    idx = np.arange(len(labels))
    outer = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for k, (tr_pool, test_idx) in enumerate(outer.split(idx, labels, groups)):
        k_val = max(2, int(round(1.0 / val_frac)))
        sub_tr, sub_val = next(StratifiedGroupKFold(
            n_splits=k_val, shuffle=True, random_state=seed + 100 + k
        ).split(tr_pool, labels[tr_pool], groups[tr_pool]))
        folds.append(dict(fold=k, train=tr_pool[sub_tr], val=tr_pool[sub_val], test=test_idx))
    return folds


def dataset_fingerprint(labels, groups, folds, cfg):
    """Hash of everything that must match for two runs to be comparable."""
    h = hashlib.sha256()
    h.update(np.asarray(labels, np.int64).tobytes())
    h.update("|".join(map(str, groups)).encode())
    for k in PROTOCOL_KEYS:
        h.update(f"{k}={cfg[k]}".encode())
    fold_of = np.full(len(labels), -1, np.int64)
    for f in folds:
        fold_of[f["test"]] = f["fold"]
    h.update(fold_of.tobytes())
    return h.hexdigest()[:16], fold_of


FOLDS = patient_level_folds(LABELS, PATIENT_ID, CFG["N_FOLDS"],
                            CFG["VAL_FRAC_OF_TRAIN"], CFG["SEED"])
FINGERPRINT, FOLD_OF = dataset_fingerprint(LABELS, PATIENT_ID, FOLDS, CFG)

print(pd.DataFrame([
    dict(fold=f["fold"],
         train_nod=len(f["train"]), val_nod=len(f["val"]), test_nod=len(f["test"]),
         train_pat=len(np.unique(PATIENT_ID[f["train"]])),
         val_pat=len(np.unique(PATIENT_ID[f["val"]])),
         test_pat=len(np.unique(PATIENT_ID[f["test"]])),
         test_pct_malig=round(100 * float(LABELS[f["test"]].mean()), 1))
    for f in FOLDS]).to_string(index=False))
print(f"\nDATASET FINGERPRINT: {FINGERPRINT}")

### Audit A1 &nbsp;|&nbsp; Fold separation

In [ ]:
print("A1.1  Patient overlap between the three splits, within each fold")
print("      (every cell must be 0)")
ok = True
for f in FOLDS:
    row = {}
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        n = len(set(PATIENT_ID[f[a]]) & set(PATIENT_ID[f[b]]))
        row[f"{a}/{b}"] = n
        ok &= (n == 0)
    print(f"      fold {f['fold']}: {row}")

print("\nA1.2  Patient overlap between the TEST folds (each patient in exactly one)")
for i in range(len(FOLDS)):
    for j in range(i + 1, len(FOLDS)):
        n = len(set(PATIENT_ID[FOLDS[i]["test"]]) & set(PATIENT_ID[FOLDS[j]["test"]]))
        ok &= (n == 0)
        if n:
            print(f"      !! folds {i},{j} share {n} patients")
print("      all pairs disjoint" if ok else "      FAILED")

print("\nA1.3  Every nodule of a patient lands in the same test fold")
per_patient = pd.DataFrame({"patient": PATIENT_ID, "fold": FOLD_OF})
spread = per_patient.groupby("patient")["fold"].nunique()
multi = per_patient.groupby("patient").size()
print(f"      patients with >1 nodule : {int((multi > 1).sum())} of {len(multi)} "
      f"(max {int(multi.max())} nodules)")
print(f"      patients split across folds: {int((spread > 1).sum())}   <- must be 0")
ok &= (spread > 1).sum() == 0

print("\nA1.4  Test folds partition the dataset exactly once")
cov = np.concatenate([f["test"] for f in FOLDS])
print(f"      test predictions {len(cov)} | unique {len(np.unique(cov))} | dataset {len(LABELS)}")
ok &= len(cov) == len(np.unique(cov)) == len(LABELS)

assert ok, "FOLD SEPARATION AUDIT FAILED - do not use these results"
print(f"\n      AUDIT A1 PASSED - folds are patient-disjoint and exhaustive")

## 5 &nbsp;Preprocessing, and what is fitted on what

Evidence for "*was preprocessing fitted only on training folds*".

The strongest possible answer is that **nothing is fitted on the data at all**, and that is the case
here. Every transform uses either a constant fixed before seeing any data, or per-scan DICOM
metadata that belongs to that scan alone. Nothing is estimated from a pooled distribution, so there
is no quantity that could carry information across the fold boundary.

| Step | Source of its parameters | Fold-dependent? |
|---|---|---|
| HU windowing [&minus;1000, 400] | fixed constant (lung window) | no |
| mm &rarr; pixel crop size | that scan's own `PixelSpacing` | no - per-scan |
| resize to 224 | fixed | no |
| scale to [0,1] | fixed &divide;255 | no |
| channel normalisation | **ImageNet** mean/std constants | no - not from LIDC |
| augmentation | fixed random policy, train split only | train only |
| class weights / sampler | counts from **that fold's training indices** | **yes - train only** |

The only fold-dependent quantity is the class balance used for the sampler, and A2 asserts it is
computed from `fold["train"]` alone. Note in particular that channel normalisation uses ImageNet
statistics rather than LIDC statistics: that is deliberate. Dataset-derived statistics would have to
be recomputed per fold to stay honest, and using fixed external constants sidesteps the issue
entirely at no measurable cost.

Augmentation is deliberately restrained - flips and 90&deg; rotations, which are safe because a
nodule has no canonical orientation, plus small intensity jitter. Aggressive intensity augmentation
would be wrong here: the input is calibrated Hounsfield Units, and distorting them destroys the
density information that separates a solid nodule from a ground-glass one.

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class NoduleDataset(Dataset):
    """Returns (image, label, cache_index). The index travels with the sample so out-of-fold
    predictions can be written back to the right row and joined to nodule metadata later."""

    def __init__(self, patches, labels, indices, train=False, seed=0):
        self.patches, self.labels, self.indices, self.train = patches, labels, indices, train
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = self.indices[i]
        x = self.patches[j].astype(np.float32) / 255.0
        if self.train:
            if self.rng.random() < .5: x = np.flip(x, axis=2)
            if self.rng.random() < .5: x = np.flip(x, axis=1)
            x = np.ascontiguousarray(np.rot90(x, int(self.rng.integers(0, 4)), axes=(1, 2)))
            if self.rng.random() < .3:
                x = np.clip(x * self.rng.uniform(.93, 1.07) + self.rng.uniform(-.04, .04), 0, 1)
        x = torch.from_numpy(np.ascontiguousarray(x))
        if x.shape[0] == 1:
            x = x.repeat(3, 1, 1)
        elif x.shape[0] > 3:
            mid = x.shape[0] // 2; x = x[mid - 1: mid + 2]
        return (x - IMAGENET_MEAN) / IMAGENET_STD, torch.tensor(int(self.labels[j])), int(j)


def make_fold_loaders(fold, cfg):
    """Only the training loader augments or rebalances; val/test see the data untouched."""
    seed = cfg["SEED"] + fold["fold"]
    tr = NoduleDataset(PATCHES, LABELS, fold["train"], train=True,  seed=seed)
    va = NoduleDataset(PATCHES, LABELS, fold["val"],   train=False, seed=seed)
    te = NoduleDataset(PATCHES, LABELS, fold["test"],  train=False, seed=seed)

    if cfg["CLASS_WEIGHTS"]:
        # counts come from this fold's TRAINING indices only - asserted in audit A2
        cnt = np.bincount(LABELS[fold["train"]], minlength=2).astype(np.float64)
        w = (1.0 / np.maximum(cnt, 1))[LABELS[fold["train"]]]
        sampler, shuffle = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                                 len(w), replacement=True), False
    else:
        sampler, shuffle = None, True

    common = dict(num_workers=cfg["NUM_WORKERS"], pin_memory=torch.cuda.is_available())
    return (DataLoader(tr, batch_size=cfg["BATCH_SIZE"], sampler=sampler, shuffle=shuffle,
                       drop_last=len(tr) > cfg["BATCH_SIZE"], **common),
            DataLoader(va, batch_size=cfg["BATCH_SIZE"] * 2, shuffle=False, **common),
            DataLoader(te, batch_size=cfg["BATCH_SIZE"] * 2, shuffle=False, **common))

### Audit A2 &nbsp;|&nbsp; Preprocessing provenance

In [ ]:
print("A2.1  Normalisation constants are external, not estimated from LIDC")
print(f"      mean {IMAGENET_MEAN.flatten().tolist()}  std {IMAGENET_STD.flatten().tolist()}")
print("      these are the published ImageNet statistics - independent of any fold")

print("\nA2.2  The only fold-dependent quantity is the sampler's class weighting")
for f in FOLDS[:2]:
    cnt_train = np.bincount(LABELS[f["train"]], minlength=2)
    cnt_all   = np.bincount(LABELS, minlength=2)
    print(f"      fold {f['fold']}: weights from train counts {cnt_train.tolist()} "
          f"(whole-dataset counts would be {cnt_all.tolist()})")

# Prove it programmatically: rebuild the weights from the training indices and confirm the
# sampler's weights match, i.e. no test or val label influenced them.
f = FOLDS[0]
tl, _, _ = make_fold_loaders(f, CFG)
if CFG["CLASS_WEIGHTS"]:
    expected = (1.0 / np.maximum(np.bincount(LABELS[f["train"]], minlength=2),
                                 1))[LABELS[f["train"]]]
    got = np.asarray(tl.sampler.weights, dtype=np.float64)
    assert np.allclose(got, expected), "sampler weights did not derive from the training split"
    print("      asserted: sampler weights reproduce exactly from fold['train'] labels only")
else:
    print("      CLASS_WEIGHTS is off - no fold-dependent quantity exists at all")

print("\nA2.3  Augmentation is applied to the training split only")
_, vl, sl = make_fold_loaders(f, CFG)
print(f"      train.train={tl.dataset.train}   val.train={vl.dataset.train}   "
      f"test.train={sl.dataset.train}")
assert tl.dataset.train and not vl.dataset.train and not sl.dataset.train
del tl, vl, sl

print("\n      AUDIT A2 PASSED - no quantity is fitted across a fold boundary")

## 6 &nbsp;Model and training

ImageNet-pretrained **ViT-tiny** (`vit_tiny_patch16_224`) with a dropout head, trained with the same
loop, optimiser, schedule and early-stopping rule as the CNN baseline.

### A trap that silently destroys Task 6's uncertainty estimates

`timm`'s `drop_rate` argument wires **only the classifier head**. Every dropout inside the
transformer blocks stays at `p = 0.0`. Passing `drop_rate` alone therefore builds a ViT with roughly
50 dropout modules that are all inert: training is unaffected, so nothing looks wrong, but Task 6's
MC-Dropout would be sampling a single linear layer and reporting its near-zero variance as
confidence.

`proj_drop_rate` is the argument that actually reaches `attn.proj_drop` and `mlp.drop1/drop2` in
every block. The cell below passes both, and then **counts the dropout layers with `p > 0`** and
prints the number. Expect roughly 20-40. If it prints 1, the uncertainty work downstream would be
meaningless, so the count is a check rather than a decoration.

`attn_drop` is deliberately left at 0: dropping attention weights perturbs the attention
distribution itself, which is not the perturbation MC Dropout's theory assumes.

### Why dropout is set during training, not added later

MC Dropout approximates a Bayesian posterior only if the network learned to be robust to its own
units being masked. Bolting dropout onto a trained deterministic network and sampling it yields
noise, not calibrated uncertainty. Setting `DROPOUT_P` here avoids retraining for Task 6.

### On hyperparameters (review question A5)

No search was run. The primary settings are copied unchanged from Task 04-01 - that is the point of
the comparison - and the tuned variant's settings (`VIT_LR`, `VIT_WARMUP_EPOCHS`) are ordinary
published defaults for fine-tuning a ViT, fixed before the first run. Neither was adjusted in
response to a result, so there is no search over test performance that could leak.

Within a run the epoch is selected on **validation AUROC** from that fold's own validation slice,
never the test fold.

In [ ]:
try:
    import timm
except ImportError:
    get_ipython().system("pip install -q timm")
    import timm
print("timm", timm.__version__)

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, brier_score_loss, precision_recall_curve)

class ViTNodule(nn.Module):
    """ImageNet-pretrained ViT with dropout that is actually active inside the blocks."""

    def __init__(self, n_classes=2, pretrained=True, p=0.3,
                 backbone="vit_tiny_patch16_224", img_size=224):
        super().__init__()
        self.backbone_name = backbone
        kw = dict(pretrained=pretrained, num_classes=0,     # num_classes=0 -> pooled features
                  drop_rate=p * 0.5, attn_drop_rate=0.0, img_size=img_size)
        try:
            # proj_drop_rate is the one that reaches attn.proj_drop and mlp.drop1/drop2
            self.body = timm.create_model(backbone, proj_drop_rate=p * 0.5, **kw)
        except TypeError:
            # older timm without the argument - set the block rates by hand instead
            self.body = timm.create_model(backbone, **kw)
            for nm, mod in self.body.named_modules():
                if isinstance(mod, nn.Dropout) and ("mlp.drop" in nm or "proj_drop" in nm):
                    mod.p = p * 0.5
        self.head_drop = nn.Dropout(p)
        self.fc = nn.Linear(self.body.num_features, n_classes)

    def forward(self, x):
        return self.fc(self.head_drop(self.body(x)))


def count_active_dropout(model):
    """Dropout modules with p > 0. A module with p == 0 injects nothing, so counting every
    Dropout instance would hide exactly the timm behaviour described above."""
    return sum(1 for m in model.modules()
               if isinstance(m, (nn.Dropout, nn.Dropout1d, nn.Dropout2d, nn.Dropout3d,
                                 nn.AlphaDropout)) and m.p > 0)


def build_model(cfg):
    return ViTNodule(2, cfg["PRETRAINED"], cfg["DROPOUT_P"],
                     cfg["VIT_BACKBONE"], cfg["IMG_SIZE"]).to(DEVICE)


_m = build_model(dict(CFG, PRETRAINED=False))
_act = count_active_dropout(_m)
_inert = sum(1 for mod in _m.modules() if isinstance(mod, nn.Dropout) and mod.p == 0)
with torch.no_grad():
    _out = _m(torch.zeros(2, 3, CFG["IMG_SIZE"], CFG["IMG_SIZE"]).to(DEVICE))
print(f"{CFG['VIT_BACKBONE']}: {sum(q.numel() for q in _m.parameters())/1e6:.2f}M params | "
      f"{_act} ACTIVE dropout ({_inert} inert) | output {tuple(_out.shape)}")
if _act < 3:
    raise RuntimeError(
        f"Only {_act} active dropout layers - proj_drop_rate did not take effect. Task 6's "
        f"MC Dropout would be meaningless on this model. Check the timm version.")
print(f"reference: ResNet-18 in Task 04-01 has 11.18M params and 5 active dropout layers")
del _m


@torch.no_grad()
def predict(model, loader):
    model.eval()
    P, Y, J = [], [], []
    for x, y, j in loader:
        P.append(torch.softmax(model(x.to(DEVICE)).float(), 1)[:, 1].cpu().numpy())
        Y.append(y.numpy()); J.append(np.asarray(j))
    return np.concatenate(P), np.concatenate(Y), np.concatenate(J)


def make_scheduler(opt, cfg, lr, warmup_epochs):
    """Cosine decay, optionally preceded by a linear warmup (the tuned ViT variant)."""
    if warmup_epochs <= 0:
        return CosineAnnealingLR(opt, T_max=cfg["EPOCHS"])
    total = max(cfg["EPOCHS"], warmup_epochs + 1)
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return (ep + 1) / warmup_epochs
        prog = (ep - warmup_epochs) / max(total - warmup_epochs, 1)
        return 0.5 * (1.0 + math.cos(math.pi * prog))
    return LambdaLR(opt, lr_lambda)


def train_one_fold(cfg, fold, train_loader, val_loader, lr=None, warmup_epochs=0, tag=""):
    """Train one fold; select the epoch on validation AUROC only."""
    set_seed(cfg["SEED"] + fold["fold"])
    model = build_model(cfg)
    lr = cfg["LR"] if lr is None else lr

    if cfg["CLASS_WEIGHTS"]:
        cnt = np.bincount(LABELS[fold["train"]], minlength=2).astype(np.float32)
        w = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
        criterion = nn.CrossEntropyLoss(weight=w)
    else:
        criterion = nn.CrossEntropyLoss()

    opt = AdamW(model.parameters(), lr=lr, weight_decay=cfg["WEIGHT_DECAY"])
    sched = make_scheduler(opt, cfg, lr, warmup_epochs)
    use_amp = cfg["AMP"] and DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ckpt = f"{cfg['OUT_DIR']}/{cfg['MODEL']}{tag}_fold{fold['fold']}_best.pt"

    best, best_ep, bad, hist = -np.inf, -1, 0, []
    t0 = time.time()
    for ep in range(1, cfg["EPOCHS"] + 1):
        model.train(); tot = seen = 0
        for x, y, _ in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * x.size(0); seen += x.size(0)
        sched.step()

        pv, yv, _ = predict(model, val_loader)
        auc = roc_auc_score(yv, pv) if len(np.unique(yv)) > 1 else np.nan
        hist.append(dict(fold=fold["fold"], epoch=ep, train_loss=tot / max(seen, 1),
                         val_auroc=float(auc), lr=opt.param_groups[0]["lr"]))
        if np.isfinite(auc) and auc > best + 1e-5:
            best, best_ep, bad = float(auc), ep, 0
            torch.save(dict(model=model.state_dict(), cfg=cfg, name=cfg["MODEL"],
                            fold=fold["fold"], epoch=ep, val_auroc=best,
                            fingerprint=FINGERPRINT, build_version=BUILD_VERSION,
                            variant=tag or "primary"), ckpt)
        else:
            bad += 1
        if bad >= cfg["PATIENCE"]:
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model"])
    print(f"    best epoch {best_ep} (val AUROC {best:.4f}) | {(time.time()-t0)/60:.1f} min")
    return model, pd.DataFrame(hist), ckpt

### Audit A5 &nbsp;|&nbsp; Model selection used no test information

In [ ]:
print("A5.1  The epoch is selected on validation AUROC; the test fold is never scored in the loop")
import inspect
params = list(inspect.signature(train_one_fold).parameters)
print(f"      signature parameters: {params}")
assert not any("test" in p for p in params), "a test loader reached the training function"

print("\nA5.2  Hyperparameters were fixed a priori - no search was run")
for k in ["EPOCHS", "BATCH_SIZE", "LR", "WEIGHT_DECAY", "DROPOUT_P", "PATIENCE"]:
    print(f"      {k:14s} {CFG[k]}")
print("      No grid, no random search, no manual tuning against a reported metric.")
print("      With no search there is no selection over test performance to leak.")

print("\nA5.3  The decision threshold is chosen on validation (section 7), not on the OOF set")
print(f"      target sensitivity {CFG['TARGET_SENSITIVITY']} -> threshold picked per fold on val")
print("\n      AUDIT A5 PASSED")

## 7 &nbsp;Evaluation harness

The same metric code Tasks 04-01 and 05-01 used, reused unchanged, so the calibration numbers below are computed identically to the ones already reported. `core_metrics`, the ECE implementation and the patient-clustered bootstrap all come from here.

In [ ]:
def sensitivity_at_specificity(y, p, target):
    """Highest sensitivity subject to specificity >= target, with the threshold that achieves it.
    This is the clinical framing: fix the false-positive burden, then ask how many cancers you catch."""
    fpr, tpr, thr = roc_curve(y, p)
    ok = (1 - fpr) >= target
    if not ok.any():
        return np.nan, np.nan, np.nan
    i = int(np.argmax(np.where(ok, tpr, -np.inf)))
    return float(tpr[i]), float(1 - fpr[i]), float(thr[i])


def expected_calibration_error(y, p, n_bins=15):
    conf = np.where(p >= .5, p, 1 - p)
    correct = ((p >= .5).astype(int) == y).astype(float)
    edges, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for i in range(n_bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1]) if i else (conf >= edges[i]) & (conf <= edges[i + 1])
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def core_metrics(y, p, threshold=0.5, cfg=CFG):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    out = dict(n=int(len(y)), threshold=float(threshold),
               accuracy=float(accuracy_score(y, pred)),
               auroc=float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
               auprc=float(average_precision_score(y, p)),
               sensitivity=float(recall_score(y, pred, zero_division=0)),
               specificity=float(tn / (tn + fp)) if (tn + fp) else np.nan,
               precision=float(precision_score(y, pred, zero_division=0)),
               f1=float(f1_score(y, pred, zero_division=0)),
               ece=expected_calibration_error(y, p, cfg["N_ECE_BINS"]),
               brier=float(brier_score_loss(y, p)),
               tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
               fnr=float(fn / (fn + tp)) if (fn + tp) else np.nan)
    for ts in cfg["TARGET_SPECIFICITIES"]:
        s, _, t = sensitivity_at_specificity(y, p, ts)
        out[f"sens_at_spec_{int(ts*100)}"] = s
        out[f"thr_at_spec_{int(ts*100)}"] = t
    return out


def threshold_for_sensitivity(y, p, target):
    """Highest threshold that still reaches the target sensitivity. Chosen on VALIDATION."""
    cand = np.unique(np.round(np.linspace(.01, .99, 99), 4))
    ok = [t for t in cand if recall_score(y, (p >= t).astype(int), zero_division=0) >= target]
    return float(max(ok)) if ok else 0.5


def bootstrap_ci(y, p, groups, fn, n_boot, seed, cluster=True):
    """Bootstrap CI. cluster=True resamples PATIENTS (correct); False resamples nodules (naive)."""
    rng = np.random.default_rng(seed)
    vals = []
    if cluster:
        uniq = np.unique(groups)
        by = {g: np.where(groups == g)[0] for g in uniq}
        for _ in range(n_boot):
            idx = np.concatenate([by[g] for g in rng.choice(uniq, len(uniq), replace=True)])
            if len(np.unique(y[idx])) > 1:
                try: vals.append(fn(y[idx], p[idx]))
                except Exception: pass
    else:
        n = len(y)
        for _ in range(n_boot):
            idx = rng.integers(0, n, n)
            if len(np.unique(y[idx])) > 1:
                try: vals.append(fn(y[idx], p[idx]))
                except Exception: pass
    if not vals:
        return np.nan, np.nan
    v = np.asarray(vals, float)
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

---

## 8.0 &nbsp;What MC Dropout produces

Keep dropout active at inference and run `T` forward passes. Each pass is a draw from an
approximate posterior over weights, so instead of one probability per nodule you get a
distribution over `T` of them. Three quantities come out, and they are **not** interchangeable:

| Quantity | Formula | Reads as |
|---|---|---|
| **Predictive entropy** | `H(mean_t p_t)` | total uncertainty |
| **Expected entropy** | `mean_t H(p_t)` | **aleatoric** &mdash; the image is ambiguous; more compute will not help |
| **Mutual information** | `H(mean) - mean H` | **epistemic** &mdash; the model does not know; more modelling may help |

All three are divided by `ln 2`, so each lands in [0, 1] for this binary task.

Note the arithmetic, which is easy to get backwards: **aleatoric is the average of the entropies;
total is the entropy of the average.** Epistemic is what is left over.

### Two implementation details that quietly ruin this

**`model.train()` is the wrong switch.** It also puts BatchNorm into training mode, which recomputes
batch statistics from the inference batch &mdash; predictions would then depend on which other
nodules share the batch. `enable_mc_dropout()` flips **only** dropout layers.

**Dropout layers must have `p > 0`, not merely exist.** A `Dropout(p=0)` sits in the graph and
injects nothing, so an existence check would pass on a model whose `T` passes are all identical.
Only layers with `p > 0` are counted, and the run refuses below a minimum.

In [ ]:
from scipy import stats          # Spearman, for the reader-agreement check in 8.4

DROPOUT_TYPES = (nn.Dropout, nn.Dropout1d, nn.Dropout2d, nn.Dropout3d, nn.AlphaDropout)

def enable_mc_dropout(model, min_active=3):
    """Activate dropout for sampling while keeping BatchNorm/LayerNorm in eval mode."""
    model.eval()                                    # everything deterministic first...
    active = [m for m in model.modules() if isinstance(m, DROPOUT_TYPES) and m.p > 0]
    for m in active:                                # ...then re-enable real dropout only
        m.train()
    if len(active) < min_active:
        raise RuntimeError(
            f"Only {len(active)} dropout layer(s) with p > 0 - MC Dropout would return nearly "
            f"identical passes and the 'uncertainty' would be an artefact. The model must have "
            f"been TRAINED with dropout active; it cannot be added afterwards.")
    return len(active)


@torch.no_grad()
def mc_dropout_predict(model, loader, n_passes, seed=0, label=""):
    """T stochastic forward passes -> (T, N, 2) probabilities, plus labels and cache indices."""
    n_active = enable_mc_dropout(model)
    torch.manual_seed(seed)
    all_passes, Y, J = [], None, None
    t0 = time.time()
    for t in range(n_passes):
        P, ys, js = [], [], []
        for x, y, j in loader:
            P.append(torch.softmax(model(x.to(DEVICE)).float(), 1).cpu().numpy())
            if t == 0:
                ys.append(y.numpy()); js.append(np.asarray(j))
        all_passes.append(np.concatenate(P, axis=0))
        if t == 0:
            Y, J = np.concatenate(ys), np.concatenate(js)
    model.eval()
    probs = np.stack(all_passes, axis=0)
    spread = float(probs[:, :, 1].std(axis=0).mean())
    print(f"      MC {label}: {n_passes} passes x {probs.shape[1]} nodules | {n_active} active "
          f"dropout | mean spread {spread:.5f} | {time.time()-t0:.1f}s")
    if spread < 1e-3:
        print("      WARNING: passes are nearly identical - uncertainty below is NOT meaningful.")
    return probs, Y, J


UNC_KEYS = ["mean_prob_positive", "confidence", "predictive_entropy", "aleatoric",
            "epistemic", "std_positive", "vote_disagreement"]

def uncertainty_from_passes(probs, eps=1e-12):
    """(T, N, C) -> per-sample decomposition, entropies normalised to [0, 1]."""
    mean_p = probs.mean(axis=0)
    total     = -(mean_p * np.log(mean_p + eps)).sum(axis=1)          # H(mean)
    aleatoric = -(probs * np.log(probs + eps)).sum(axis=2).mean(0)    # mean H
    ln2 = np.log(2.0)
    votes = (probs[:, :, 1] >= 0.5).mean(axis=0)
    return dict(
        mean_prob_positive = mean_p[:, 1],
        confidence         = mean_p.max(axis=1),
        predictive_entropy = total / ln2,
        aleatoric          = aleatoric / ln2,
        epistemic          = np.clip(total - aleatoric, 0.0, None) / ln2,
        std_positive       = probs[:, :, 1].std(axis=0),
        vote_disagreement  = np.minimum(votes, 1.0 - votes) * 2.0,
    )

## 8.1 &nbsp;Finding and verifying the checkpoints

Every checkpoint records the dataset fingerprint it was trained under. That is checked against this
session before anything expensive runs.

For the ViT arm there is a trap worth naming: the Task 05-01 output contains **ten** `.pt` files,
five primary and five from the tuned-variant sensitivity check. Only the primary set belongs here.
The selection below is explicit about which files it took and which it ignored.

In [ ]:
def find_fold_checkpoints(model_name, cfg):
    """One checkpoint per fold. Excludes variant checkpoints such as *_tuned_*."""
    found, ignored = {}, []
    for root in [cfg["OUT_DIR"], cfg["WORK"], "/kaggle/input"]:
        for p in sorted(glob.glob(f"{root}/**/*_best.pt", recursive=True)):
            base = os.path.basename(p)
            if not base.startswith(f"{model_name}_fold"):
                if base.startswith(model_name):
                    ignored.append(base)
                continue
            try:
                k = int(base.split("_fold")[1].split("_")[0])
            except (IndexError, ValueError):
                continue
            found.setdefault(k, p)
    return found, sorted(set(ignored))


def load_fold_model(path, cfg, expect_fingerprint=None):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    got = ck.get("fingerprint")
    if expect_fingerprint is not None and got is not None and got != expect_fingerprint:
        raise RuntimeError(
            f"FINGERPRINT MISMATCH on {os.path.basename(path)}\n"
            f"  checkpoint trained on : {got}\n"
            f"  this session's data   : {expect_fingerprint}\n"
            f"These nodules are NOT out-of-fold for this model, so any uncertainty computed from "
            f"it would be invalid. Attach the same cache the checkpoints were trained on.")
    model = build_model(cfg)
    model.load_state_dict(ck["model"])
    model.eval()
    return model, ck


CKPTS_BY_FOLD, IGNORED = find_fold_checkpoints(CFG["MODEL"], CFG)
print(f"looking for {CFG['MODEL']}_fold*_best.pt")
for k in sorted(CKPTS_BY_FOLD):
    print(f"  fold {k}: {CKPTS_BY_FOLD[k]}")
if IGNORED:
    print(f"  ignored (not the primary set): {IGNORED}")

missing = [k for k in range(CFG["N_FOLDS"]) if k not in CKPTS_BY_FOLD]
if missing:
    raise FileNotFoundError(
        f"Missing checkpoints for fold(s) {missing}.\n"
        f"Attach the {CFG['MODEL']} run's output: Add Input -> Your Work / Notebook Output, or a "
        f"Dataset containing {CFG['MODEL']}_fold0_best.pt ... _fold{CFG['N_FOLDS']-1}_best.pt.")

for k in sorted(CKPTS_BY_FOLD):
    _m, _ck = load_fold_model(CKPTS_BY_FOLD[k], CFG, expect_fingerprint=FINGERPRINT)
    print(f"  fold {k}: epoch {_ck.get('epoch','?')}, val AUROC "
          f"{_ck.get('val_auroc', float('nan')):.4f}, fingerprint {_ck.get('fingerprint','?')}  OK")
    del _m
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"\nAll {CFG['N_FOLDS']} checkpoints present and consistent with {FINGERPRINT}")

## 8.2 &nbsp;Out-of-fold MC Dropout

For each fold: load its checkpoint, run one deterministic pass (the Task 4/5 baseline prediction)
and `T` stochastic passes over that fold's held-out nodules, then write both into full-dataset
vectors.

The fold's **validation** slice is sampled too, but used only to calibrate thresholds in 8.5.
Those thresholds must never be chosen on the out-of-fold set the results are reported from.

In [ ]:
def mc_dropout_over_folds(cfg, folds, ckpts):
    N = len(LABELS)
    oof = dict(p_det=np.full(N, np.nan), fold=np.full(N, -1, dtype=np.int64))
    for k in UNC_KEYS:
        oof[k] = np.full(N, np.nan)
    val_pool, per_fold = [], []
    t0 = time.time()

    for f in folds:
        k = f["fold"]
        print(f"\n--- fold {k} ---")
        model, _ = load_fold_model(ckpts[k], cfg, expect_fingerprint=FINGERPRINT)
        tl, vl, sl = make_fold_loaders(f, cfg)

        p_det, y_t, j_t = predict(model, sl)          # deterministic = the Task 4/5 behaviour
        oof["p_det"][j_t] = p_det
        oof["fold"][j_t] = k

        probs, _, j_mc = mc_dropout_predict(model, sl, cfg["MC_PASSES"], cfg["SEED"] + k, "test")
        u = uncertainty_from_passes(probs)
        for key in UNC_KEYS:
            oof[key][j_mc] = u[key]

        probs_v, y_v, j_v = mc_dropout_predict(model, vl, cfg["MC_PASSES"],
                                               cfg["SEED"] + 500 + k, "val ")
        uv = uncertainty_from_passes(probs_v)
        val_pool.append(dict(y=y_v, j=j_v, **{key: uv[key] for key in UNC_KEYS}))

        m_det = core_metrics(LABELS[j_t], p_det, 0.5, cfg)
        m_mc  = core_metrics(LABELS[j_mc], u["mean_prob_positive"], 0.5, cfg)
        per_fold.append(dict(fold=k, auroc_det=m_det["auroc"], auroc_mc=m_mc["auroc"],
                             ece_det=m_det["ece"], ece_mc=m_mc["ece"],
                             brier_det=m_det["brier"], brier_mc=m_mc["brier"],
                             acc_det=m_det["accuracy"], acc_mc=m_mc["accuracy"],
                             fn_det=m_det["fn"], fn_mc=m_mc["fn"]))
        print(f"      fold {k}: AUROC {m_det['auroc']:.4f} -> {m_mc['auroc']:.4f} | "
              f"ECE {m_det['ece']:.4f} -> {m_mc['ece']:.4f} | "
              f"Brier {m_det['brier']:.4f} -> {m_mc['brier']:.4f}")

        del model, tl, vl, sl
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    for key in ["p_det"] + UNC_KEYS:
        assert not np.isnan(oof[key]).any(), f"{key} has gaps - a fold did not cover its nodules"
    print(f"\nMC dropout complete in {(time.time()-t0)/60:.1f} min")

    # Validation slices come from each fold's TRAINING pool, so a nodule can appear in more than
    # one fold's validation set. Fine for choosing a threshold; not an unbiased performance set.
    VAL = {key: np.concatenate([v[key] for v in val_pool]) for key in UNC_KEYS}
    VAL["y"] = np.concatenate([v["y"] for v in val_pool])
    VAL["j"] = np.concatenate([v["j"] for v in val_pool])
    print(f"pooled validation for calibration: {len(VAL['y'])} predictions "
          f"({len(np.unique(VAL['j']))} distinct nodules)")
    return oof, VAL, pd.DataFrame(per_fold)


OOF, VAL, MC_FOLD_TABLE = mc_dropout_over_folds(CFG, FOLDS, CKPTS_BY_FOLD)
P_DET, P_MC = OOF["p_det"], OOF["mean_prob_positive"]

## 8.3 &nbsp;Calibration &mdash; the deliverable

The task asks for **ECE and Brier**. Both are reported for the deterministic baseline and the
MC-averaged prediction, per fold and pooled, so the effect of MC Dropout is measured rather than
asserted.

- **ECE** isolates calibration: when the model says 0.8, is it right 80% of the time?
- **Brier** is a proper scoring rule mixing calibration *and* discrimination.
- **AUROC** is a control: averaging over passes should not cost discrimination.

The expected pattern is that MC averaging **improves ECE and Brier** and leaves AUROC roughly
unchanged. That &mdash; calibrated confidence rather than higher accuracy &mdash; is MC Dropout's
real benefit. If ECE gets *worse*, the dropout rate is too high, and that gets reported as a
finding rather than hidden.

In [ ]:
print(f"PER-FOLD: deterministic vs MC dropout ({CFG['MODEL']})")
print(MC_FOLD_TABLE.round(4).to_string(index=False))
print("\nmean +/- SD across folds")
for a, b, nm in [("ece_det","ece_mc","ECE"), ("brier_det","brier_mc","Brier"),
                 ("auroc_det","auroc_mc","AUROC"), ("acc_det","acc_mc","Accuracy")]:
    d, m = MC_FOLD_TABLE[a], MC_FOLD_TABLE[b]
    better = (m.mean() < d.mean()) if nm in ("ECE", "Brier") else (m.mean() > d.mean())
    print(f"  {nm:9s} deterministic {d.mean():.4f} +/- {d.std(ddof=1):.4f}   "
          f"MC {m.mean():.4f} +/- {m.std(ddof=1):.4f}   ({'improved' if better else 'worse'})")

print("\n" + "=" * 76)
print(f"POOLED OUT-OF-FOLD - {len(LABELS)} nodules, {len(np.unique(PATIENT_ID))} patients"
      + ("   [SMOKE - NOT REPORTABLE]" if IS_SMOKE else ""))
print("=" * 76)

UQ_RESULTS = []
for tag, p in [("deterministic", P_DET), (f"mcdropout_T{CFG['MC_PASSES']}", P_MC)]:
    m = core_metrics(LABELS, p, 0.5, CFG)
    m.update(model=CFG["MODEL"], tag=tag, smoke=IS_SMOKE, fingerprint=FINGERPRINT)
    for key, fn in [("auroc", roc_auc_score),
                    ("ece", lambda a, b: expected_calibration_error(a, b, CFG["N_ECE_BINS"])),
                    ("brier", brier_score_loss)]:
        lo, hi = bootstrap_ci(LABELS, p, PATIENT_ID, fn, CFG["N_BOOTSTRAP"], CFG["SEED"])
        m[f"{key}_ci_low"], m[f"{key}_ci_high"] = lo, hi
    UQ_RESULTS.append(m)
    print(f"\n  {tag}")
    print(f"    ECE    {m['ece']:.4f}  [{m['ece_ci_low']:.4f}, {m['ece_ci_high']:.4f}]")
    print(f"    Brier  {m['brier']:.4f}  [{m['brier_ci_low']:.4f}, {m['brier_ci_high']:.4f}]")
    print(f"    AUROC  {m['auroc']:.4f}  [{m['auroc_ci_low']:.4f}, {m['auroc_ci_high']:.4f}]")
    print(f"    acc {m['accuracy']:.4f}  sens {m['sensitivity']:.4f}  spec {m['specificity']:.4f}"
          f"  FN {m['fn']}")

d, mc = UQ_RESULTS[0], UQ_RESULTS[1]
print("\n" + "-" * 76)
print(f"  ECE    {d['ece']:.4f} -> {mc['ece']:.4f}   "
      f"({'improved' if mc['ece'] < d['ece'] else 'WORSE - dropout rate may be too high'})")
print(f"  Brier  {d['brier']:.4f} -> {mc['brier']:.4f}   "
      f"({'improved' if mc['brier'] < d['brier'] else 'worse'})")
print(f"  AUROC  {d['auroc']:.4f} -> {mc['auroc']:.4f}   (control - should be ~unchanged)")
print("-" * 76)

## 8.4 &nbsp;Is the uncertainty worth acting on?

Three checks, each able to fail. An uncertainty number that fails them is decorative.

1. **Does uncertainty rank the model's own errors?** Treat "was this prediction wrong" as the
   label and the uncertainty as the score. 0.5 means the signal is noise.
2. **Does referral beat random referral?** Any referral policy produces a falling
   risk&ndash;coverage curve on an imbalanced problem. The gap against a random-referral control is
   the actual finding.
3. **Does the decomposition mean what it claims?** LIDC gives 3&ndash;4 independent malignancy
   ratings per nodule, so `mal_std` is *measured* human ambiguity. If the **aleatoric** term
   correlates with it more strongly than the **epistemic** term does, the split is behaving as the
   theory says. No published work appears to have run this test on LIDC.

In [ ]:
correct = (P_MC >= 0.5).astype(int) == LABELS
U = OOF["predictive_entropy"]
ERROR_RANK_AUROC = float(roc_auc_score((~correct).astype(int), U))
print(f"1. Uncertainty ranks its own errors: AUROC = {ERROR_RANK_AUROC:.4f}")
print(f"   (0.5 = noise, must not drive routing;  >0.7 = usable signal)")

def risk_coverage(y, p, unc, threshold=0.5, n_points=50):
    order = np.argsort(unc)
    y_s, pred_s = y[order], (p[order] >= threshold).astype(int)
    n = len(y); covs, errs, fnrs = [], [], []
    for c in np.linspace(max(0.1, 5.0 / n), 1.0, n_points):
        k = max(1, int(round(c * n))); yy, pp = y_s[:k], pred_s[:k]
        covs.append(k / n); errs.append(float((yy != pp).mean()))
        pos = yy == 1
        fnrs.append(float(((pp == 0) & pos).sum() / max(pos.sum(), 1)))
    covs, errs, fnrs = map(np.asarray, (covs, errs, fnrs))
    integrate = getattr(np, "trapezoid", None) or np.trapz
    return covs, errs, fnrs, float(integrate(errs, covs) / (covs[-1] - covs[0]))

covs, errs, fnrs, AURC = risk_coverage(LABELS, P_MC, U)
rng = np.random.default_rng(CFG["SEED"])
rand = np.mean([risk_coverage(LABELS, P_MC, rng.permutation(U))[1] for _ in range(30)], axis=0)
integrate = getattr(np, "trapezoid", None) or np.trapz
AURC_RAND = float(integrate(rand, covs) / (covs[-1] - covs[0]))
print(f"\n2. AURC by uncertainty {AURC:.4f}  vs random referral {AURC_RAND:.4f}  "
      f"(gain {AURC_RAND - AURC:+.4f})")
for c in [1.0, 0.9, 0.8, 0.7]:
    k = int(round(c * len(LABELS))); o = np.argsort(U)[:k]
    print(f"     coverage {c:5.0%}  error {(((P_MC[o]>=.5).astype(int))!=LABELS[o]).mean():.4f}"
          f"   FN {int(((P_MC[o]<.5)&(LABELS[o]==1)).sum()):3d}")

READER_CORR = pd.DataFrame()
if len(META) and "mal_std" in META.columns:
    rs = META["mal_std"].to_numpy(dtype=float)
    if np.std(rs) > 1e-9:
        rows = []
        for key in ["aleatoric", "epistemic", "predictive_entropy"]:
            rho, pv = stats.spearmanr(rs, OOF[key])
            rows.append(dict(measure=key, spearman_rho=rho, p_value=pv))
        READER_CORR = pd.DataFrame(rows)
        print(f"\n3. Spearman correlation with radiologist disagreement (n={len(rs)}):")
        for r in READER_CORR.itertuples():
            print(f"     {r.measure:20s} rho = {r.spearman_rho:+.3f}   p = {r.p_value:.4g}")
        ra = READER_CORR.loc[READER_CORR.measure=="aleatoric","spearman_rho"].iloc[0]
        re_ = READER_CORR.loc[READER_CORR.measure=="epistemic","spearman_rho"].iloc[0]
        print(f"     aleatoric {ra:+.3f} vs epistemic {re_:+.3f} -> "
              + ("AS PREDICTED: aleatoric tracks human ambiguity more strongly"
                 if ra > re_ else
                 "NOT as predicted - report honestly; it weakens the decomposition claim"))
else:
    print("\n3. Reader-disagreement test needs mal_std in the cache metadata - unavailable.")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))

for lab, p, col in [("deterministic", P_DET, "#A3323C"),
                    (f"MC T={CFG['MC_PASSES']}", P_MC, "#1B4F9C")]:
    ece = expected_calibration_error(LABELS, p, CFG["N_ECE_BINS"])
    conf = np.where(p >= .5, p, 1 - p); corr = ((p >= .5).astype(int) == LABELS).astype(float)
    edges = np.linspace(0, 1, CFG["N_ECE_BINS"] + 1); xs, ys = [], []
    for i in range(CFG["N_ECE_BINS"]):
        m = (conf > edges[i]) & (conf <= edges[i+1]) if i else (conf >= edges[i]) & (conf <= edges[i+1])
        if m.any(): xs.append(conf[m].mean()); ys.append(corr[m].mean())
    ax[0].plot(xs, ys, "o-", ms=4, color=col, label=f"{lab}  ECE {ece:.3f}")
ax[0].plot([0,1],[0,1],"k:",lw=1); ax[0].set_xlabel("predicted confidence")
ax[0].set_ylabel("observed accuracy"); ax[0].set_title("Reliability"); ax[0].legend(fontsize=8)

for msk, lab, col in [(correct,"correct","#1F7A5C"), (~correct,"incorrect","#A3323C")]:
    ax[1].scatter(OOF["aleatoric"][msk], OOF["epistemic"][msk], s=15, alpha=.6, label=lab, color=col)
ax[1].set_xlabel("aleatoric (image ambiguity)"); ax[1].set_ylabel("epistemic (model ignorance)")
ax[1].set_title("Uncertainty decomposition"); ax[1].legend(fontsize=8)

ax[2].plot(covs, errs, lw=2, color="#1B4F9C", label=f"by uncertainty (AURC {AURC:.4f})")
ax[2].plot(covs, rand, ls=":", lw=1.4, color="grey", label=f"random (AURC {AURC_RAND:.4f})")
ax[2].set_xlabel("coverage"); ax[2].set_ylabel("error rate")
ax[2].set_title("Risk-coverage"); ax[2].legend(fontsize=8)

ax[3].hist([U[correct], U[~correct]], bins=20, density=True,
           color=["#1F7A5C","#A3323C"], label=["correct","incorrect"])
ax[3].set_xlabel("predictive entropy"); ax[3].set_ylabel("density")
ax[3].set_title(f"Errors rank at AUROC {ERROR_RANK_AUROC:.3f}"); ax[3].legend(fontsize=8)

fig.suptitle(f"Task 6  |  {CFG['MODEL']}  MC Dropout T={CFG['MC_PASSES']}  |  "
             f"n={len(LABELS)}, {len(np.unique(PATIENT_ID))} patients"
             + ("   [SMOKE]" if IS_SMOKE else ""), fontsize=12)
plt.tight_layout()
plt.savefig(f"{CFG['OUT_DIR']}/task06_{CFG['MODEL']}_results.png", dpi=120, bbox_inches="tight")
plt.show()

## 8.5 &nbsp;Stage 2 &mdash; threshold routing, no LLM

This is the **control the orchestrator has to beat**. Two thresholds, calibrated on the validation
slices, produce the three routing outcomes. It is deterministic, free, and needs no network.

If an LLM routes no better than this, it is adding latency, cost and a hallucination risk for
nothing &mdash; so this stage must exist before stage 3 can be claimed as a contribution.

In [ ]:
def calibrate_uq_thresholds(val, target_sensitivity, max_referral=0.30):
    """Pick the decision and abstention thresholds on pooled VALIDATION predictions only."""
    y, p, u = val["y"], val["mean_prob_positive"], val["predictive_entropy"]
    cand = np.unique(np.round(np.linspace(.02, .98, 97), 4))
    sens = np.array([recall_score(y, (p >= t).astype(int), zero_division=0) for t in cand])
    ok = np.where(sens >= target_sensitivity)[0]
    decision = float(cand[ok[-1]]) if len(ok) else 0.5

    pred = (p >= decision).astype(int)
    fn_mask = (pred == 0) & (y == 1)
    base_fn = int(fn_mask.sum())
    best_q, best_rate = float(1 - max_referral), float(max_referral)
    for q in np.arange(0.95, 1.0 - max_referral - 1e-9, -0.01):
        keep = u <= float(np.quantile(u, q))
        if base_fn == 0 or int((fn_mask & keep).sum()) <= base_fn * 0.5:
            best_q, best_rate = float(q), float((~keep).mean()); break

    return dict(decision_threshold=decision,
                uncertainty_threshold=float(np.quantile(u, best_q)),
                epistemic_threshold=float(np.quantile(val["epistemic"], best_q)),
                calibrated_on="pooled per-fold validation slices",
                target_sensitivity=float(target_sensitivity),
                expected_referral_rate=best_rate)


def route_by_threshold(entropy, epistemic, thr):
    """accept | escalate_model | human_review - the deterministic policy."""
    if entropy <= thr["uncertainty_threshold"]:
        return "accept"
    return "escalate_model" if epistemic >= thr["epistemic_threshold"] else "human_review"


def score_policy(actions, y, p, decision_threshold, name):
    """Selective-prediction view: what happens to the cases a policy keeps?"""
    a = np.asarray(actions)
    kept = a == "accept"
    pred = (p >= decision_threshold).astype(int)
    row = dict(policy=name, n=len(y),
               accept=int(kept.sum()), coverage=float(kept.mean()),
               escalate_model=int((a == "escalate_model").sum()),
               human_review=int((a == "human_review").sum()),
               fn_kept=int(((pred == 0) & (y == 1) & kept).sum()),
               fn_total=int(((pred == 0) & (y == 1)).sum()),
               err_kept=float((pred[kept] != y[kept]).mean()) if kept.any() else np.nan,
               err_all=float((pred != y).mean()))
    row["sens_kept"] = (float(recall_score(y[kept], pred[kept], zero_division=0))
                        if kept.any() and len(np.unique(y[kept])) > 1 else np.nan)
    return row


UQ_THRESHOLDS = calibrate_uq_thresholds(VAL, CFG["UQ_TARGET_SENSITIVITY"])
print("Thresholds (calibrated on validation only):")
for k, v in UQ_THRESHOLDS.items():
    print(f"  {k:26s} {v}")

THR_ACTIONS = [route_by_threshold(OOF["predictive_entropy"][i], OOF["epistemic"][i],
                                  UQ_THRESHOLDS) for i in range(len(LABELS))]
POLICY_ROWS = [score_policy(["accept"] * len(LABELS), LABELS, P_MC,
                            UQ_THRESHOLDS["decision_threshold"], "no routing (accept all)"),
               score_policy(THR_ACTIONS, LABELS, P_MC,
                            UQ_THRESHOLDS["decision_threshold"], "stage 2: thresholds")]
print("\n" + pd.DataFrame(POLICY_ROWS).round(4).to_string(index=False))

## 9 &nbsp;Stage 3 &mdash; the Qwen3 orchestrator (optional)

Qwen3 reads the uncertainty record and chooses the same three actions. It is reached through
**OpenRouter**, so nothing is downloaded and no GPU is involved &mdash; it is an HTTPS call.

### Supplying the API key

The cell tries three sources in order, and **skips the whole section cleanly** if none is
available, so the Task 6 deliverable is never blocked:

1. **Kaggle Secrets** &mdash; `Add-ons -> Secrets`, add `OPENROUTER_API_KEY`, attach it to this
   notebook. This is the only option that also works when you **Save Version**, because a committed
   run has no one to type at.
2. **Environment variable** `OPENROUTER_API_KEY`.
3. **An interactive prompt** &mdash; a hidden input box appears when you run the cell. Convenient
   while experimenting; it cannot work in a committed run, and the cell says so rather than hanging.

Never paste the key into a cell. If the notebook is ever made public the key goes with it.

### Four rules this implementation follows

- **The model never sees the label.** The prompt carries uncertainty and nodule geometry only.
- **Every call is logged verbatim** to `llm_calls.jsonl` and **cached by prompt hash**. LLM outputs
  are not reproducible by re-running, so the log *is* the record &mdash; and the cache makes a
  Save Version re-run replay exactly what you saw, instead of re-querying and possibly differing.
- **`temperature = 0`** and a pinned model id.
- **It decides, it does not predict.** The malignancy probability always comes from the CNN. An LLM
  asserting a diagnosis from numbers is the documented hallucination risk.

In [ ]:
# ------------------------------------------------------------------ key + client
def get_openrouter_key():
    """Kaggle Secrets -> env var -> interactive prompt. Returns (key, source) or (None, reason)."""
    try:
        from kaggle_secrets import UserSecretsClient
        k = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
        if k:
            return k.strip(), "kaggle secrets"
    except Exception:
        pass
    if os.environ.get("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"].strip(), "environment variable"
    try:
        import getpass
        k = getpass.getpass("OpenRouter API key (blank to skip the LLM section): ")
        if k.strip():
            return k.strip(), "interactive prompt"
        return None, "left blank"
    except Exception as e:
        # a committed / batch run has no stdin - skip rather than hang
        return None, f"no interactive input available ({type(e).__name__})"


LLM_MODEL = "qwen/qwen3-8b"          # pinned; see the plan document for the cost comparison
LLM_LOG   = f"{CFG['OUT_DIR']}/llm_calls.jsonl"

SYSTEM_PROMPT = """You are a triage router in a lung-nodule screening pipeline. You never diagnose.
A convolutional network has already produced a malignancy probability and an uncertainty
decomposition for one nodule. Your only job is to choose what happens to that case next.

Choose exactly one action:
  accept          - the prediction stands, no further work
  escalate_model  - re-analyse with test-time augmentation or the fold ensemble
  human_review    - send to a radiologist

Guidance:
- Low total uncertainty means accept.
- High uncertainty that is mostly EPISTEMIC means the model is outside what it learned, so more
  modelling may help -> escalate_model.
- High uncertainty that is mostly ALEATORIC means the image itself is ambiguous, so more compute
  will not resolve it -> human_review.
- A missed cancer is far costlier than a false alarm.

Reply with JSON only, no prose: {"action": "...", "reason": "<one short sentence>"}"""


def build_prompt(i):
    """Uncertainty and geometry only. The label is never included."""
    d = {
        "p_malignant": round(float(P_MC[i]), 4),
        "total_uncertainty": round(float(OOF["predictive_entropy"][i]), 4),
        "aleatoric": round(float(OOF["aleatoric"][i]), 4),
        "epistemic": round(float(OOF["epistemic"][i]), 4),
        "pass_disagreement": round(float(OOF["vote_disagreement"][i]), 4),
        "thresholds": {k: round(v, 4) for k, v in UQ_THRESHOLDS.items()
                       if isinstance(v, float)},
    }
    if len(META) and "nodule_w_mm" in META.columns:
        d["nodule_size_mm"] = round(float(max(META.nodule_w_mm.iloc[i],
                                              META.nodule_h_mm.iloc[i])), 1)
    return json.dumps(d)


LLM_KEY, LLM_SOURCE = (None, "disabled") if not CFG["USE_LLM"] else get_openrouter_key()

if LLM_KEY is None:
    print(f"LLM section SKIPPED - {LLM_SOURCE}.")
    print("Everything above is complete; Task 6's deliverable does not depend on this section.")
    LLM_ACTIONS = None
else:
    try:
        from openai import OpenAI
    except ImportError:
        get_ipython().system("pip install -q openai")
        from openai import OpenAI
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=LLM_KEY)
    print(f"OpenRouter client ready | key from {LLM_SOURCE} | model {LLM_MODEL}")

    _cache = {}
    if os.path.exists(LLM_LOG):
        for line in open(LLM_LOG):
            try:
                r = json.loads(line); _cache[r["key"]] = r["response"]
            except Exception:
                continue
        print(f"replayed {len(_cache)} cached decisions from a previous run")

    def ask_orchestrator(i):
        prompt = build_prompt(i)
        key = hashlib.sha256((LLM_MODEL + SYSTEM_PROMPT + prompt).encode()).hexdigest()[:16]
        if key in _cache:
            return _cache[key]
        r = client.chat.completions.create(
            model=LLM_MODEL, temperature=0,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": prompt}])
        out = r.choices[0].message.content
        with open(LLM_LOG, "a") as fh:
            fh.write(json.dumps(dict(key=key, model=LLM_MODEL, prompt=prompt, response=out,
                                     usage=r.usage.model_dump() if r.usage else None)) + "\n")
        _cache[key] = out
        return out

    VALID = {"accept", "escalate_model", "human_review"}
    def parse_action(text, fallback):
        """Tolerant parse of the model's reply.

        Qwen3 is a hybrid-thinking family and some variants emit <think>...</think> before the
        answer, so reasoning is stripped first. Then the LAST flat {...} object is taken: a greedy
        match would run from a brace inside the reasoning to the final brace and never parse.

        An unreadable reply falls back to the THRESHOLD decision, not to "accept" - failing open
        on a screening task is the wrong default.
        """
        t = re.sub(r"<think>.*?</think>", " ", text, flags=re.S | re.I)
        for cand in reversed(re.findall(r"\{[^{}]*\}", t)):
            try:
                a = str(json.loads(cand)["action"]).strip().lower()
                if a in VALID:
                    return a, True
            except Exception:
                continue
        # last resort: the action word appearing bare in the text
        for a in VALID:
            if re.search(rf"\b{a}\b", t, re.I):
                return a, True
        return fallback, False

    n = min(len(LABELS), CFG["LLM_MAX_CALLS"])
    LLM_ACTIONS, n_bad, n_new = list(THR_ACTIONS), 0, 0
    print(f"routing {n} nodules through {LLM_MODEL} ({len(_cache)} already cached)")
    t0 = time.time()
    for i in range(n):
        was_cached = True
        key_probe = hashlib.sha256((LLM_MODEL + SYSTEM_PROMPT
                                    + build_prompt(i)).encode()).hexdigest()[:16]
        if key_probe not in _cache:
            was_cached, n_new = False, n_new + 1
        a, ok = parse_action(ask_orchestrator(i), THR_ACTIONS[i])
        LLM_ACTIONS[i] = a
        n_bad += (not ok)
        # first live call tells you the real latency, so the wait is predictable
        if not was_cached and n_new == 1:
            print(f"  first live call took {time.time()-t0:.1f}s -> roughly "
                  f"{(time.time()-t0)*(n-i)/60:.1f} min for the rest at this rate")
        if (i + 1) % 25 == 0:
            el = time.time() - t0
            print(f"  {i+1}/{n} | {el:.0f}s elapsed | {el/(i+1):.2f}s per decision | "
                  f"eta {el/(i+1)*(n-i-1)/60:.1f} min")
    el = time.time() - t0
    print(f"\n{n} decisions in {el:.0f}s ({el/max(n,1):.2f}s each) | {n_new} live calls, "
          f"{n - n_new} replayed from cache")
    print(f"unparseable replies: {n_bad} (fell back to the threshold rule) | "
          f"log -> {os.path.basename(LLM_LOG)}")
    if n_bad > n * 0.1:
        print("NOTE: many replies failed to parse. If the model is emitting long reasoning, try a "
              "non-thinking variant or lower LLM_MAX_CALLS and inspect llm_calls.jsonl.")

## 10 &nbsp;Did the LLM route better than the thresholds?

Both policies are scored the same way, on the same out-of-fold nodules. The question is not whether
the LLM is clever, but whether the cases it keeps are safer than the cases the threshold rule keeps
&mdash; at comparable coverage.

Read `fn_kept` first: false negatives among the auto-decided cases are the ones that would reach a
patient uncorrected.

In [ ]:
if LLM_ACTIONS is not None:
    POLICY_ROWS.append(score_policy(LLM_ACTIONS, LABELS, P_MC,
                                    UQ_THRESHOLDS["decision_threshold"],
                                    f"stage 3: {LLM_MODEL}"))
POLICY_TABLE = pd.DataFrame(POLICY_ROWS)
print(POLICY_TABLE.round(4).to_string(index=False))

if LLM_ACTIONS is not None:
    same = sum(a == b for a, b in zip(THR_ACTIONS, LLM_ACTIONS))
    print(f"\nAgreement with the threshold rule: {same}/{len(LABELS)} ({same/len(LABELS):.1%})")
    print("\nWhere they differ:")
    from collections import Counter
    diff = Counter((t, l) for t, l in zip(THR_ACTIONS, LLM_ACTIONS) if t != l)
    for (t, l), c in diff.most_common():
        print(f"  threshold={t:15s} -> llm={l:15s}  {c}")

    t_row = POLICY_TABLE[POLICY_TABLE.policy == "stage 2: thresholds"].iloc[0]
    l_row = POLICY_TABLE.iloc[-1]
    print(f"\n  coverage  {t_row.coverage:.1%} -> {l_row.coverage:.1%}")
    print(f"  FN kept   {int(t_row.fn_kept)} -> {int(l_row.fn_kept)}")
    print(f"  err kept  {t_row.err_kept:.4f} -> {l_row.err_kept:.4f}")
    if l_row.fn_kept <= t_row.fn_kept and l_row.coverage >= t_row.coverage:
        print("  -> the LLM is at least as safe at no less coverage: a real gain")
    elif l_row.coverage < t_row.coverage and l_row.fn_kept < t_row.fn_kept:
        print("  -> fewer missed cancers, but it refers more cases: a trade, not a free win")
    else:
        print("  -> no advantage over the deterministic rule. Report that: on this dataset the")
        print("     orchestrator does not earn its cost, which is a legitimate finding.")
else:
    print("\n(stage 3 not run - the threshold policy is the only routing result)")

## 11 &nbsp;Outputs

Five files, plus the LLM log when stage 3 ran. `uq_interface_contract.json` is the deliverable to
send Dr. Sattar &mdash; the file itself, not a description of it.

**Save Version before closing the session**, or `/kaggle/working` is discarded.

In [ ]:
OUT, M = CFG["OUT_DIR"], CFG["MODEL"]

np.savez_compressed(f"{OUT}/task06_{M}_mc_oof.npz",
                    p_det=P_DET, p_mc=P_MC, y=LABELS,
                    cache_index=np.arange(len(LABELS)), patient_id=PATIENT_ID.astype(object),
                    fold=OOF["fold"], malignancy=MALIGNANCY,
                    **{k: OOF[k] for k in UNC_KEYS},
                    model=M, fingerprint=FINGERPRINT, mc_passes=CFG["MC_PASSES"])

pd.DataFrame(UQ_RESULTS).to_csv(f"{OUT}/task06_{M}_uq_metrics.csv", index=False)
MC_FOLD_TABLE.to_csv(f"{OUT}/task06_{M}_fold_metrics.csv", index=False)
POLICY_TABLE.to_csv(f"{OUT}/task06_{M}_routing.csv", index=False)

d, mc = UQ_RESULTS[0], UQ_RESULTS[1]
CONTRACT = dict(
    schema_version="uq-1.1",
    produced_by=f"Task 6 - MC Dropout on {M}",
    dataset_fingerprint=FINGERPRINT, smoke=bool(IS_SMOKE),
    n_nodules=int(len(LABELS)), n_patients=int(len(np.unique(PATIENT_ID))),
    entry_point="records carry probabilities + decomposed uncertainty + a routing hint",
    input_preprocessing={k: CFG[k] for k in ["PATCH_MM", "IMG_SIZE", "N_SLICES", "SLICE_MM",
                                             "HU_WINDOW", "MIN_READERS", "DROP_SCORE_3"]},
    record_fields={
        "p_malignant": "float - MC-averaged over T passes",
        "predictive_entropy": "float [0,1] - TOTAL uncertainty",
        "aleatoric": "float [0,1] - image ambiguity; escalation will NOT help",
        "epistemic": "float [0,1] - model ignorance; escalation MAY help",
        "vote_disagreement": "float [0,1] - 0 all passes agree, 1 even split",
        "routing_hint": "accept | escalate_model | human_review (ADVISORY)",
    },
    routing_semantics={
        "accept": "total uncertainty below threshold - use the prediction",
        "escalate_model": "uncertain AND epistemic-dominated - TTA or the 5-fold ensemble",
        "human_review": "uncertain AND aleatoric-dominated - a radiologist; compute will not help",
    },
    calibrated_thresholds=UQ_THRESHOLDS,
    calibration={r["tag"]: {k: r[k] for k in ["ece", "brier", "auroc", "accuracy",
                                              "sensitivity", "specificity", "fn"]}
                 for r in UQ_RESULTS},
    diagnostics=dict(uncertainty_error_ranking_auroc=ERROR_RANK_AUROC,
                     aurc_by_uncertainty=AURC, aurc_random_referral=AURC_RAND,
                     reader_agreement=(READER_CORR.to_dict("records") if len(READER_CORR) else [])),
    routing_comparison=POLICY_TABLE.to_dict("records"),
    llm=(dict(model=LLM_MODEL, source=LLM_SOURCE, calls=int(min(len(LABELS),
         CFG["LLM_MAX_CALLS"])), log=os.path.basename(LLM_LOG))
         if LLM_ACTIONS is not None else None),
    known_limitations=[
        "binary benign/malignant; the ordinal 1-5 task is not implemented",
        f"median malignancy == 3 excluded, so indeterminate lesions are unseen and unvalidated",
        f"evaluated on {len(LABELS)} nodules from {len(np.unique(PATIENT_ID))} patients - "
        f"confidence intervals are wide at this size",
        f"MC Dropout costs {CFG['MC_PASSES']}x a forward pass",
        "the ViT failed the complementarity gate, so escalation targets TTA and the fold "
        "ensemble, not an alternative architecture",
    ],
)
with open(f"{OUT}/uq_interface_contract.json", "w") as fh:
    json.dump(CONTRACT, fh, indent=2, default=str)

print(f"OUTPUTS -> {OUT}")
for p in sorted(glob.glob(f"{OUT}/*")):
    print(f"  {os.path.getsize(p)/1024:9.1f} KB  {os.path.basename(p)}")

print(f"""
{'='*76}
TASK 6  |  {M}  |  T={CFG['MC_PASSES']}  |  fingerprint {FINGERPRINT}
{'='*76}
  CALIBRATION (the deliverable), pooled out-of-fold
    ECE     deterministic {d['ece']:.4f}   ->   MC {mc['ece']:.4f}
    Brier   deterministic {d['brier']:.4f}   ->   MC {mc['brier']:.4f}
    AUROC   deterministic {d['auroc']:.4f}   ->   MC {mc['auroc']:.4f}   (control)

  DIAGNOSTICS
    uncertainty ranks its own errors   AUROC {ERROR_RANK_AUROC:.4f}
    AURC by uncertainty {AURC:.4f}  vs random {AURC_RAND:.4f}

CHECKLIST
  [{'X' if not IS_SMOKE else ' '}] real LIDC data (SMOKE is {IS_SMOKE})
  [{'X' if ERROR_RANK_AUROC > 0.7 else ' '}] uncertainty ranks errors above 0.70
  [{'X' if AURC < AURC_RAND else ' '}] beats random referral
  [ ] uq_interface_contract.json sent to Dr. Sattar
  [ ] Save Version before closing
{'='*76}
""")